In [1]:

# SEAG qualification period is 22 Oct 2024 to 5 Sept 2025

#1. Segregate into Male and Femal 
#2. For each gender perform the following: 
#a. Sort data by mapped eent, then perf scalar (higher the better)
#b. Identify tiers based on performance - Tier 1 is meets bronze medal mark for SEAG, Tier 2 is 2% and Tier 3 is 3.5%
#c. Check - if athlete met bronze or 2%/3.5% then delta_benchmark is zero or +, delta2% is + and delta 3.5% is +
#d. Top ranked athletes for each event are chosen. Max number of athletes for each event is 3, except for 100m/400m which is 6
#    This includes athletes on spex scholarship and potential
#e. The max for each tier is 2. Lower ranked athletes move down one tier.
#3. If athlete qualifies for more than one event the higher tier event is given
#4. Jump and throws junior program to be solved separately

%load_ext autoreload
%autoreload 2

In [2]:
# Import usual modules
import pandas as pd
import csv
import math
import os
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import openpyxl
import datetime
from scipy.stats import lognorm
import re
import string
from bs4 import BeautifulSoup
import requests
import unicodedata # for removing accented characters
import datetime
import icecream as ic
import dateutil.parser as parser 
import datacompy
import pytz
import gspread

from google.cloud import storage



Please note that you are missing the optional dependency: fugue. If you need to use this functionality it must be installed.
Please note that you are missing the optional dependency: spark. If you need to use this functionality it must be installed.
Please note that you are missing the optional dependency: snowflake. If you need to use this functionality it must be installed.


In [3]:
# PRODUCTION ENVIRONMENT
# Extract timed event records

import pandas_gbq
from google.oauth2 import service_account

credentials = service_account.Credentials.from_service_account_file(
    '/Users/veesheenyuen/Desktop/DataScience/Keys/saa-analytics-7c8937b70609.json',
    
    
)

sql1="""
SELECT NAME, RESULT, TEAM, AGE, RANK AS COMPETITION_RANK, DIVISION, EVENT, DISTANCE, EVENT_CLASS, UNIQUE_ID, DOB, NATIONALITY, WIND, CATEGORY_EVENT, GENDER, COMPETITION, DATE, YEAR, REGION, TIMESTAMP
FROM `saa-analytics.results.PRODUCTION_CORRECT` 
WHERE RESULT!='NM' AND RESULT!='-' AND RESULT!='DNS' AND RESULT!='DNF' AND RESULT!='DNQ' AND RESULT!='DQ' AND RESULT IS NOT NULL

"""

competitors = pandas_gbq.read_gbq(sql1, project_id="saa-analytics", credentials=credentials)




Downloading: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████|


In [4]:
competitors

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT,DISTANCE,EVENT_CLASS,UNIQUE_ID,DOB,NATIONALITY,WIND,CATEGORY_EVENT,GENDER,COMPETITION,DATE,YEAR,REGION,TIMESTAMP
0,SI EN TABITHA NG,00:11:03.950000,,,4.0,,3000m,,,,,SGP,,Long,Female,14TH ASEAN SCHOOLS GAMES,2025-11-23 00:00:00+00:00,2025,International,2025-12-09 12:38:00+00:00
1,JE AN GARRETT CHUA,6.84,,,3.0,,Long Jump,,,,,SGP,1.0,Jump,Male,14TH ASEAN SCHOOLS GAMES,2025-11-23 00:00:00+00:00,2025,International,2025-12-09 12:38:00+00:00
2,LAUREL JIA EN LIM,27.71,,,6.0,,Discus Throw,,,,,SGP,,Throw,Female,14TH ASEAN SCHOOLS GAMES,2025-11-23 00:00:00+00:00,2025,International,2025-12-09 12:38:00+00:00
3,JOSHUA SHYEN LEE,11.03,,,4.0,,100m,,,,,SGP,-0.5,Sprint,Male,14TH ASEAN SCHOOLS GAMES,2025-11-23 00:00:00+00:00,2025,International,2025-12-09 12:38:00+00:00
4,DARYEN XIN TZE KO,54.65,,,2.0,,400m Hurdles,,,,,SGP,,Hurdles,Male,14TH ASEAN SCHOOLS GAMES,2025-11-23 00:00:00+00:00,2025,International,2025-12-09 12:38:00+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
170101,"YONG HUI, LAI",12.79,ACTIVESG ATHLETICS,34,nan,nan,Triple Jump,,,,1989,,0,Jump,Male,Club Zoom,2023-11-25 00:00:00+00:00,2023,Local,2025-12-07 17:55:00+00:00
170102,"CHEN, KE YUAN",11.72,Nanyang Technological Uni,23,nan,nan,Triple Jump,,,,2000,,0,Jump,Male,SA - 5th All Comers,2023-12-09 00:00:00+00:00,2023,nan,2025-12-07 17:55:00+00:00
170103,"YEOH, YUE QI",8.39,Singapore Management Uni,19,nan,nan,Triple Jump,,,,2004,,0,Jump,Female,SA - 5th All Comers,2023-12-09 00:00:00+00:00,2023,nan,2025-12-07 17:55:00+00:00
170104,"TAN, TSE TENG",10.9,Nanyang Technological Uni,21,nan,nan,Triple Jump,,,,2002,,0,Jump,Female,SA - 5th All Comers,2023-12-09 00:00:00+00:00,2023,nan,2025-12-07 17:55:00+00:00


In [5]:
competitors[competitors['COMPETITION']=='The 7th Tokai Sprint Games']

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT,DISTANCE,EVENT_CLASS,UNIQUE_ID,DOB,NATIONALITY,WIND,CATEGORY_EVENT,GENDER,COMPETITION,DATE,YEAR,REGION,TIMESTAMP
27369,Ryan Praharsh,10.70,,,8,,100m,,,,18 Oct 98,SGP,0.1,Sprint,Male,The 7th Tokai Sprint Games,2025-08-17 00:00:00+00:00,2025,International,2025-11-17 16:42:00+00:00
150727,Ryan Praharsh,10.70,,,5,,100m,,,,18 Oct 98,SGP,0.0,Sprint,Male,The 7th Tokai Sprint Games,2025-08-17 00:00:00+00:00,2025,International,2025-11-17 16:42:00+00:00


In [6]:
def convert_time_format(time_str):
    """
    Convert time from 'HH:MM:SS.mmmmmm' to 'MM:SS.mm' format.
    Also formats float values to ensure 2 decimal places (e.g., 9.1 -> 9.10).
    
    Args:
        time_str: Time string in format 'HH:MM:SS.mmmmmm' or float value
    
    Returns:
        Converted time string in format 'MM:SS.mm' or formatted float with 2 decimals
    """
    if pd.isna(time_str):
        return time_str
    
    time_str = str(time_str)
    
    # Match pattern HH:MM:SS.mmmmmm (with flexible microseconds)
    pattern = r'^(\d{2}):(\d{2}):(\d{2})\.(\d+)$'
    match = re.match(pattern, time_str)
    
    if match:
        hours, minutes, seconds, microseconds = match.groups()
        # Take only first 2 digits of microseconds (centiseconds)
        centiseconds = microseconds[:2].ljust(2, '0')
        return f"{minutes}:{seconds}.{centiseconds}"
    
    # Check if it's a float value (e.g., 9.1, 12.34)
    try:
        float_val = float(time_str)
        return f"{float_val:.2f}"
    except ValueError:
        pass
    
    # Return original if pattern doesn't match and not a float
    return time_str

competitors['RESULT'] = competitors['RESULT'].apply(convert_time_format)


In [7]:
os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/Asian Indoor/')


competitors.to_csv('database_prod_test_Jan3.csv', sep=',', encoding='utf-8-sig', index=False)

In [8]:
competitors

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT,DISTANCE,EVENT_CLASS,UNIQUE_ID,DOB,NATIONALITY,WIND,CATEGORY_EVENT,GENDER,COMPETITION,DATE,YEAR,REGION,TIMESTAMP
0,SI EN TABITHA NG,11:03.95,,,4.0,,3000m,,,,,SGP,,Long,Female,14TH ASEAN SCHOOLS GAMES,2025-11-23 00:00:00+00:00,2025,International,2025-12-09 12:38:00+00:00
1,JE AN GARRETT CHUA,6.84,,,3.0,,Long Jump,,,,,SGP,1.0,Jump,Male,14TH ASEAN SCHOOLS GAMES,2025-11-23 00:00:00+00:00,2025,International,2025-12-09 12:38:00+00:00
2,LAUREL JIA EN LIM,27.71,,,6.0,,Discus Throw,,,,,SGP,,Throw,Female,14TH ASEAN SCHOOLS GAMES,2025-11-23 00:00:00+00:00,2025,International,2025-12-09 12:38:00+00:00
3,JOSHUA SHYEN LEE,11.03,,,4.0,,100m,,,,,SGP,-0.5,Sprint,Male,14TH ASEAN SCHOOLS GAMES,2025-11-23 00:00:00+00:00,2025,International,2025-12-09 12:38:00+00:00
4,DARYEN XIN TZE KO,54.65,,,2.0,,400m Hurdles,,,,,SGP,,Hurdles,Male,14TH ASEAN SCHOOLS GAMES,2025-11-23 00:00:00+00:00,2025,International,2025-12-09 12:38:00+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
170101,"YONG HUI, LAI",12.79,ACTIVESG ATHLETICS,34,nan,nan,Triple Jump,,,,1989,,0,Jump,Male,Club Zoom,2023-11-25 00:00:00+00:00,2023,Local,2025-12-07 17:55:00+00:00
170102,"CHEN, KE YUAN",11.72,Nanyang Technological Uni,23,nan,nan,Triple Jump,,,,2000,,0,Jump,Male,SA - 5th All Comers,2023-12-09 00:00:00+00:00,2023,nan,2025-12-07 17:55:00+00:00
170103,"YEOH, YUE QI",8.39,Singapore Management Uni,19,nan,nan,Triple Jump,,,,2004,,0,Jump,Female,SA - 5th All Comers,2023-12-09 00:00:00+00:00,2023,nan,2025-12-07 17:55:00+00:00
170104,"TAN, TSE TENG",10.90,Nanyang Technological Uni,21,nan,nan,Triple Jump,,,,2002,,0,Jump,Female,SA - 5th All Comers,2023-12-09 00:00:00+00:00,2023,nan,2025-12-07 17:55:00+00:00


In [9]:
competitors[competitors['COMPETITION']=='The 7th Tokai Sprint Games']

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT,DISTANCE,EVENT_CLASS,UNIQUE_ID,DOB,NATIONALITY,WIND,CATEGORY_EVENT,GENDER,COMPETITION,DATE,YEAR,REGION,TIMESTAMP
27369,Ryan Praharsh,10.70,,,8,,100m,,,,18 Oct 98,SGP,0.1,Sprint,Male,The 7th Tokai Sprint Games,2025-08-17 00:00:00+00:00,2025,International,2025-11-17 16:42:00+00:00
150727,Ryan Praharsh,10.70,,,5,,100m,,,,18 Oct 98,SGP,0.0,Sprint,Male,The 7th Tokai Sprint Games,2025-08-17 00:00:00+00:00,2025,International,2025-11-17 16:42:00+00:00


In [10]:
competitors[competitors['COMPETITION']=='The 7th Tokai Sprint Games']

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT,DISTANCE,EVENT_CLASS,UNIQUE_ID,DOB,NATIONALITY,WIND,CATEGORY_EVENT,GENDER,COMPETITION,DATE,YEAR,REGION,TIMESTAMP
27369,Ryan Praharsh,10.70,,,8,,100m,,,,18 Oct 98,SGP,0.1,Sprint,Male,The 7th Tokai Sprint Games,2025-08-17 00:00:00+00:00,2025,International,2025-11-17 16:42:00+00:00
150727,Ryan Praharsh,10.70,,,5,,100m,,,,18 Oct 98,SGP,0.0,Sprint,Male,The 7th Tokai Sprint Games,2025-08-17 00:00:00+00:00,2025,International,2025-11-17 16:42:00+00:00


In [11]:
'''
# Calculate number of days from today to event date

competitors['event_date_dt'] = pd.to_datetime(competitors['event_date'], format='mixed', dayfirst=False)

competitors['delta_time']= datetime.datetime.now() - competitors['event_date_dt']

competitors['delta_time_conv'] = pd.to_numeric(competitors['delta_time'].dt.days, downcast='integer')

competitors['event_month'] = competitors['event_date_dt'].dt.month
'''

"\n# Calculate number of days from today to event date\n\ncompetitors['event_date_dt'] = pd.to_datetime(competitors['event_date'], format='mixed', dayfirst=False)\n\ncompetitors['delta_time']= datetime.datetime.now() - competitors['event_date_dt']\n\ncompetitors['delta_time_conv'] = pd.to_numeric(competitors['delta_time'].dt.days, downcast='integer')\n\ncompetitors['event_month'] = competitors['event_date_dt'].dt.month\n"

In [12]:
# DATE column to contain timezone - tz aware mode

competitors['DATE'] = pd.to_datetime(competitors['DATE'], format='mixed', dayfirst=False, utc=True)


In [13]:
# datetime to contain UTC (timezone)

competitors['NOW'] = datetime.datetime.now()

timezone = pytz.timezone('UTC')

competitors['NOW'] = datetime.datetime.now().replace(tzinfo=timezone)

In [14]:
# Calculate number of days from today to event date

#competitors['DATE'] = pd.to_datetime(competitors['DATE'], format='mixed', dayfirst=False, utc=False)

competitors['delta_time'] = competitors['NOW'] - competitors['DATE']


#competitors['delta_time'] = datetime.datetime.now() - competitors['DATE']


competitors['delta_time_conv'] = pd.to_numeric(competitors['delta_time'].dt.days, downcast='integer')

competitors['event_month'] = competitors['DATE'].dt.month

# Make sure date conversion is is valid for all rows

assert not competitors['delta_time'].isna().any()

In [15]:
competitors[competitors['COMPETITION']=='The 7th Tokai Sprint Games']

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,GENDER,COMPETITION,DATE,YEAR,REGION,TIMESTAMP,NOW,delta_time,delta_time_conv,event_month
27369,Ryan Praharsh,10.70,,,8,,100m,,,,...,Male,The 7th Tokai Sprint Games,2025-08-17 00:00:00+00:00,2025,International,2025-11-17 16:42:00+00:00,2026-01-03 16:46:53.731392+00:00,139 days 16:46:53.731392,139,8
150727,Ryan Praharsh,10.70,,,5,,100m,,,,...,Male,The 7th Tokai Sprint Games,2025-08-17 00:00:00+00:00,2025,International,2025-11-17 16:42:00+00:00,2026-01-03 16:46:53.731392+00:00,139 days 16:46:53.731392,139,8


In [16]:
# Choose date range for qualification window from 1 Jan 25 to 20 Dec 25

competitors['DATE']=competitors['DATE'].dt.tz_localize(None)  # switch off timezone for compatibility with np.datetime64

start = datetime.datetime(2025, 1, 1)

end = datetime.datetime(2025, 12, 31)

start_date = np.datetime64(start)
end_date = np.datetime64(end)


mask = (competitors['DATE'] >= start_date) & (competitors['DATE'] <= end_date)
athletes_selected = competitors.loc[mask]



In [17]:
end_date - start_date

numpy.timedelta64(31449600000000,'us')

In [18]:
athletes_selected[athletes_selected['COMPETITION']=='The 7th Tokai Sprint Games']

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,GENDER,COMPETITION,DATE,YEAR,REGION,TIMESTAMP,NOW,delta_time,delta_time_conv,event_month
27369,Ryan Praharsh,10.70,,,8,,100m,,,,...,Male,The 7th Tokai Sprint Games,2025-08-17,2025,International,2025-11-17 16:42:00+00:00,2026-01-03 16:46:53.731392+00:00,139 days 16:46:53.731392,139,8
150727,Ryan Praharsh,10.70,,,5,,100m,,,,...,Male,The 7th Tokai Sprint Games,2025-08-17,2025,International,2025-11-17 16:42:00+00:00,2026-01-03 16:46:53.731392+00:00,139 days 16:46:53.731392,139,8


In [19]:
os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/Asian Indoor/')


athletes_selected.to_csv('dates_selected_jan3.csv', encoding='utf-8')

In [20]:
athletes_selected

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,GENDER,COMPETITION,DATE,YEAR,REGION,TIMESTAMP,NOW,delta_time,delta_time_conv,event_month
0,SI EN TABITHA NG,11:03.95,,,4.0,,3000m,,,,...,Female,14TH ASEAN SCHOOLS GAMES,2025-11-23,2025,International,2025-12-09 12:38:00+00:00,2026-01-03 16:46:53.731392+00:00,41 days 16:46:53.731392,41,11
1,JE AN GARRETT CHUA,6.84,,,3.0,,Long Jump,,,,...,Male,14TH ASEAN SCHOOLS GAMES,2025-11-23,2025,International,2025-12-09 12:38:00+00:00,2026-01-03 16:46:53.731392+00:00,41 days 16:46:53.731392,41,11
2,LAUREL JIA EN LIM,27.71,,,6.0,,Discus Throw,,,,...,Female,14TH ASEAN SCHOOLS GAMES,2025-11-23,2025,International,2025-12-09 12:38:00+00:00,2026-01-03 16:46:53.731392+00:00,41 days 16:46:53.731392,41,11
3,JOSHUA SHYEN LEE,11.03,,,4.0,,100m,,,,...,Male,14TH ASEAN SCHOOLS GAMES,2025-11-23,2025,International,2025-12-09 12:38:00+00:00,2026-01-03 16:46:53.731392+00:00,41 days 16:46:53.731392,41,11
4,DARYEN XIN TZE KO,54.65,,,2.0,,400m Hurdles,,,,...,Male,14TH ASEAN SCHOOLS GAMES,2025-11-23,2025,International,2025-12-09 12:38:00+00:00,2026-01-03 16:46:53.731392+00:00,41 days 16:46:53.731392,41,11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
154444,"Ng, Zavier",9.88m,Hwa Chong Institution,14,18,U15,Triple Jump,0.0,None,Z854G11,...,Male,SA Allcomers Meet 2,2025-03-02,2025,Local,2025-11-15 13:47:37.773700,2026-01-03 16:46:53.731392+00:00,307 days 16:46:53.731392,307,3
154445,"Choo Jia Yi, Allyson",9.30m,Team Start Singapore,17.0,3,Open,Triple Jump,0.0,None,A967D08,...,Female,SA Allcomers Meet 3,2025-08-30,2025,Local,2025-11-15 13:49:23.923169,2026-01-03 16:46:53.731392+00:00,126 days 16:46:53.731392,126,8
154450,Lua Yu Xuan,10.02,NYGH,,1.0,C,Triple Jump,,,,...,Female,National School Games,2025-04-18,2025,Local,2025-11-17 17:34:00+00:00,2026-01-03 16:46:53.731392+00:00,260 days 16:46:53.731392,260,4
154451,Muhammad Aaryan Shah Bin Azhar,12.61,SSP,,3.0,B,Triple Jump,,,,...,Male,National School Games,2025-04-13,2025,Local,2025-11-17 17:34:00+00:00,2026-01-03 16:46:53.731392+00:00,265 days 16:46:53.731392,265,4


In [21]:
athletes_selected[athletes_selected['COMPETITION']=='The 7th Tokai Sprint Games']

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,GENDER,COMPETITION,DATE,YEAR,REGION,TIMESTAMP,NOW,delta_time,delta_time_conv,event_month
27369,Ryan Praharsh,10.70,,,8,,100m,,,,...,Male,The 7th Tokai Sprint Games,2025-08-17,2025,International,2025-11-17 16:42:00+00:00,2026-01-03 16:46:53.731392+00:00,139 days 16:46:53.731392,139,8
150727,Ryan Praharsh,10.70,,,5,,100m,,,,...,Male,The 7th Tokai Sprint Games,2025-08-17,2025,International,2025-11-17 16:42:00+00:00,2026-01-03 16:46:53.731392+00:00,139 days 16:46:53.731392,139,8


In [22]:
# Choose 2024/25 only

athletes = athletes_selected

In [23]:
athletes

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,GENDER,COMPETITION,DATE,YEAR,REGION,TIMESTAMP,NOW,delta_time,delta_time_conv,event_month
0,SI EN TABITHA NG,11:03.95,,,4.0,,3000m,,,,...,Female,14TH ASEAN SCHOOLS GAMES,2025-11-23,2025,International,2025-12-09 12:38:00+00:00,2026-01-03 16:46:53.731392+00:00,41 days 16:46:53.731392,41,11
1,JE AN GARRETT CHUA,6.84,,,3.0,,Long Jump,,,,...,Male,14TH ASEAN SCHOOLS GAMES,2025-11-23,2025,International,2025-12-09 12:38:00+00:00,2026-01-03 16:46:53.731392+00:00,41 days 16:46:53.731392,41,11
2,LAUREL JIA EN LIM,27.71,,,6.0,,Discus Throw,,,,...,Female,14TH ASEAN SCHOOLS GAMES,2025-11-23,2025,International,2025-12-09 12:38:00+00:00,2026-01-03 16:46:53.731392+00:00,41 days 16:46:53.731392,41,11
3,JOSHUA SHYEN LEE,11.03,,,4.0,,100m,,,,...,Male,14TH ASEAN SCHOOLS GAMES,2025-11-23,2025,International,2025-12-09 12:38:00+00:00,2026-01-03 16:46:53.731392+00:00,41 days 16:46:53.731392,41,11
4,DARYEN XIN TZE KO,54.65,,,2.0,,400m Hurdles,,,,...,Male,14TH ASEAN SCHOOLS GAMES,2025-11-23,2025,International,2025-12-09 12:38:00+00:00,2026-01-03 16:46:53.731392+00:00,41 days 16:46:53.731392,41,11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
154444,"Ng, Zavier",9.88m,Hwa Chong Institution,14,18,U15,Triple Jump,0.0,None,Z854G11,...,Male,SA Allcomers Meet 2,2025-03-02,2025,Local,2025-11-15 13:47:37.773700,2026-01-03 16:46:53.731392+00:00,307 days 16:46:53.731392,307,3
154445,"Choo Jia Yi, Allyson",9.30m,Team Start Singapore,17.0,3,Open,Triple Jump,0.0,None,A967D08,...,Female,SA Allcomers Meet 3,2025-08-30,2025,Local,2025-11-15 13:49:23.923169,2026-01-03 16:46:53.731392+00:00,126 days 16:46:53.731392,126,8
154450,Lua Yu Xuan,10.02,NYGH,,1.0,C,Triple Jump,,,,...,Female,National School Games,2025-04-18,2025,Local,2025-11-17 17:34:00+00:00,2026-01-03 16:46:53.731392+00:00,260 days 16:46:53.731392,260,4
154451,Muhammad Aaryan Shah Bin Azhar,12.61,SSP,,3.0,B,Triple Jump,,,,...,Male,National School Games,2025-04-13,2025,Local,2025-11-17 17:34:00+00:00,2026-01-03 16:46:53.731392+00:00,265 days 16:46:53.731392,265,4


In [24]:
athletes[athletes['NAME']=='Jun Jie Calvin Quek']

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,GENDER,COMPETITION,DATE,YEAR,REGION,TIMESTAMP,NOW,delta_time,delta_time_conv,event_month
61,Jun Jie Calvin Quek,50.27,<NA>,<NA>,1.0,<NA>,400m Hurdles,<NA>,<NA>,<NA>,...,Male,33rd SEA Games,2025-12-15 00:00:00,2025,International,2025-12-20 18:16:00+00:00,2026-01-03 16:46:53.731392+00:00,19 days 16:46:53.731392,19,12
462,Jun Jie Calvin Quek,10.97,-,,2,,100m,,nan,,...,Male,Pesta Sukan Athletics 2025,2025-07-26 17:45:00,2025,Local,2025-12-10 13:45:00+00:00,2026-01-03 16:46:53.731392+00:00,160 days 23:01:53.731392,160,7
530,Jun Jie Calvin Quek,11.08,-,,3,,100m,,nan,,...,Male,Pesta Sukan Athletics 2025,2025-08-02 11:55:00,2025,Local,2025-12-10 13:45:00+00:00,2026-01-03 16:46:53.731392+00:00,154 days 04:51:53.731392,154,8
983,Jun Jie Calvin Quek,47.39,-,,1,,400m,,nan,,...,Male,Pesta Sukan Athletics 2025,2025-07-27 16:40:00,2025,Local,2025-12-10 13:45:00+00:00,2026-01-03 16:46:53.731392+00:00,160 days 00:06:53.731392,160,7
1021,Jun Jie Calvin Quek,47.76,-,,1,,400m,,nan,,...,Male,Pesta Sukan Athletics 2025,2025-08-02 14:05:00,2025,Local,2025-12-10 13:45:00+00:00,2026-01-03 16:46:53.731392+00:00,154 days 02:41:53.731392,154,8
28114,Jun Jie Calvin Quek,10.86,,,2,,100m,,,,...,Male,Potch Invitational Meet,2025-04-16 00:00:00,2025,International,2025-11-17 16:42:00+00:00,2026-01-03 16:46:53.731392+00:00,262 days 16:46:53.731392,262,4
29017,Jun Jie Calvin Quek,52.62,,,9,,400m Hurdles,,,,...,Male,Sydney Track Classic,2025-03-15 00:00:00,2025,International,2025-11-17 16:42:00+00:00,2026-01-03 16:46:53.731392+00:00,294 days 16:46:53.731392,294,3
63998,Jun Jie Calvin Quek,50.58,,,2,,400m Hurdles,,,,...,Male,26th Asian Athletics Championships,2025-05-31 00:00:00,2025,International,2025-11-17 16:42:00+00:00,2026-01-03 16:46:53.731392+00:00,217 days 16:46:53.731392,217,5
65430,Jun Jie Calvin Quek,50.82,,,1,,400m Hurdles,,,,...,Male,National Track and Field Regional Invitational...,2025-07-20 00:00:00,2025,International,2025-11-17 16:42:00+00:00,2026-01-03 16:46:53.731392+00:00,167 days 16:46:53.731392,167,7
130889,Jun Jie Calvin Quek,51.64,,,6,,400m Hurdles,,,,...,Male,12th Kinami Michitaka Memorial Athletics Meet,2025-05-11 00:00:00,2025,International,2025-11-17 16:42:00+00:00,2026-01-03 16:46:53.731392+00:00,237 days 16:46:53.731392,237,5


In [25]:
# Create temporary mapped event column based on senior selection parameters

'''
athletes['MAPPED_EVENT']=''

for col in athletes.columns:
    athletes[col] = athletes[col].astype(str)
    athletes[col] = athletes[col].str.replace('\xa0', ' ', regex=True)
    athletes[col] = athletes[col].str.replace('[\x00-\x1f\x7f-\x9f]', '', regex=True)
    athletes[col] = athletes[col].str.replace('\r', ' ', regex=True)
    athletes[col] = athletes[col].str.replace('\n', ' ', regex=True)
    athletes[col] = athletes[col].str.strip()


# Correct javelin category

mask = athletes['EVENT'].str.contains(r'Javelin', na=True)
athletes.loc[mask, 'CATEGORY_EVENT'] = 'Throw'


# Running

mask = (athletes['EVENT'].str.contains(r'Dash', na=True) & athletes['DISTANCE'].str.contains(r'60', na=True))
athletes.loc[mask, 'MAPPED_EVENT'] = '60m'
mask = (athletes['EVENT'].str.contains(r'Run', na=True) & athletes['DISTANCE'].str.contains(r'60', na=True))
athletes.loc[mask, 'MAPPED_EVENT'] = '60m'
mask = athletes['EVENT'].str.contains(r'60 Meter Run', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '60m'
mask = athletes['EVENT'].str.contains(r'^60m$', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '60m'


mask = (athletes['EVENT'].str.contains(r'Dash', na=True) & athletes['DISTANCE'].str.contains(r'100', na=True))
athletes.loc[mask, 'MAPPED_EVENT'] = '100m'
mask = (athletes['EVENT'].str.contains(r'Run', na=True) & athletes['DISTANCE'].str.contains(r'100', na=True))
athletes.loc[mask, 'MAPPED_EVENT'] = '100m'
mask = athletes['EVENT'].str.contains(r'100 Meter Run', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '100m'
mask = athletes['EVENT'].str.contains(r'^100m$', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '100m'


mask = (athletes['EVENT'].str.contains(r'Dash', na=True) & athletes['DISTANCE'].str.contains(r'400', na=True))
athletes.loc[mask, 'MAPPED_EVENT'] = '400m'
mask = athletes['EVENT'].str.contains(r'^400m$', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '400m'
mask = athletes['EVENT'].str.contains(r'^400\sMeter$', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '400m'
mask = (athletes['EVENT'].str.contains(r'Run', na=True) & athletes['DISTANCE'].str.contains(r'400', na=True))
athletes.loc[mask, 'MAPPED_EVENT'] = '400m'


mask = (athletes['EVENT'].str.contains(r'Run', na=True) & athletes['DISTANCE'].str.contains(r'800', na=True))
athletes.loc[mask, 'MAPPED_EVENT'] = '800m'
mask = athletes['EVENT'].str.contains(r'800 Meter Run', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '800m'
mask = athletes['EVENT'].str.contains(r'^800m$', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '800m'
mask = (athletes['EVENT'].str.contains(r'Run', na=True) & athletes['DISTANCE'].str.contains(r'1000', na=True))
athletes.loc[mask, 'MAPPED_EVENT'] = '1000m'


mask = (athletes['EVENT'].str.contains(r'Run', na=True) & athletes['DISTANCE'].str.contains(r'1500', na=True))
athletes.loc[mask, 'MAPPED_EVENT'] = '1500m'
mask = athletes['EVENT'].str.contains(r'^1500m$', na=True, regex=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '1500m'
mask = (athletes['EVENT'].str.contains(r'Run', na=True) & athletes['DISTANCE'].str.contains(r'3000', na=True))
athletes.loc[mask, 'MAPPED_EVENT'] = '3000m'
#mask = athletes['EVENT'].str.contains(r'3000m', na=True)
#athletes.loc[mask, 'MAPPED_EVENT'] = '3000m'
mask = (athletes['EVENT'].str.contains(r'Run', na=True) & athletes['DISTANCE'].str.contains(r'5000', na=True))
athletes.loc[mask, 'MAPPED_EVENT'] = '5000m'
mask = athletes['EVENT'].str.contains(r'^5000m$', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '5000m'



# Hurdles

mask = (athletes['EVENT'].str.contains(r'60m Hurdles|60m hurdles', na=False))
athletes.loc[mask, 'MAPPED_EVENT'] = '60m Hurdles'
mask = (athletes['EVENT'].str.contains(r'^Hurdles$', na=False) & athletes['DISTANCE'].str.contains(r'60', na=False)) 
athletes.loc[mask, 'MAPPED_EVENT'] = '60m Hurdles'



mask = (athletes['EVENT'].str.contains(r'100m Hurdles|100m hurdles', na=False) & athletes['EVENT_CLASS'].str.contains('0.84|0.838|83.8', na=False) & athletes['GENDER'].str.contains(r'Female', na=False))  # this is the correct syntax
athletes.loc[mask, 'MAPPED_EVENT'] = '100m Hurdles'
mask = (athletes['EVENT'].str.contains(r'100m Hurdles|100m hurdles', na=False) & athletes['DIVISION'].str.contains('None', na=False) & athletes['GENDER'].str.contains(r'Female', na=False) & athletes['REGION'].str.contains(r'International', na=False))  # this is the correct syntax
athletes.loc[mask, 'MAPPED_EVENT'] = '100m Hurdles'
mask = (athletes['EVENT'].str.contains(r'^Hurdles$', na=False) & athletes['DISTANCE'].str.contains(r'100', na=False) & athletes['DIVISION'].str.contains(r'OPEN|Open', na=False) & athletes['GENDER'].str.contains(r'Female', na=False))
athletes.loc[mask, 'MAPPED_EVENT'] = '100m Hurdles'
mask = (athletes['EVENT'].str.contains(r'100m Hurdles|100m hurdles', na=False) & athletes['REGION'].str.contains(r'International', na=False) & athletes['GENDER'].str.contains(r'Female', na=False))
athletes.loc[mask, 'MAPPED_EVENT'] = '100m Hurdles'
mask = (athletes['EVENT'].str.contains(r'^Hurdles$', na=False) & athletes['DISTANCE'].str.contains(r'100', na=False) & athletes['EVENT_CLASS'].str.contains(r'0.838|0.84', na=False) & athletes['GENDER'].str.contains(r'Female', na=False))
athletes.loc[mask, 'MAPPED_EVENT'] = '100m Hurdles'


mask = (athletes['EVENT'].str.contains(r'100m Hurdles|100m hurdles', na=False) & athletes['EVENT_CLASS'].str.contains('0.84|0.838|83.8', na=False) & athletes['GENDER'].str.contains(r'Female', na=False))  # this is the correct syntax
athletes.loc[mask, 'MAPPED_EVENT'] = '100m Hurdles'
mask = (athletes['EVENT'].str.contains(r'100m Hurdles|100m hurdles', na=False) & athletes['DIVISION'].str.contains('None', na=False) & athletes['GENDER'].str.contains(r'Female', na=False) & athletes['REGION'].str.contains(r'International', na=False))  # this is the correct syntax
athletes.loc[mask, 'MAPPED_EVENT'] = '100m Hurdles'
mask = (athletes['EVENT'].str.contains(r'^Hurdles$', na=False) & athletes['DISTANCE'].str.contains(r'100', na=False) & athletes['DIVISION'].str.contains(r'OPEN|Open', na=False) & athletes['GENDER'].str.contains(r'Female', na=False))
athletes.loc[mask, 'MAPPED_EVENT'] = '100m Hurdles'
mask = (athletes['EVENT'].str.contains(r'100m Hurdles|100m hurdles', na=False) & athletes['REGION'].str.contains(r'International', na=False) & athletes['GENDER'].str.contains(r'Female', na=False))
athletes.loc[mask, 'MAPPED_EVENT'] = '100m Hurdles'
mask = (athletes['EVENT'].str.contains(r'^Hurdles$', na=False) & athletes['DISTANCE'].str.contains(r'100', na=False) & athletes['EVENT_CLASS'].str.contains(r'0.838|0.84', na=False) & athletes['GENDER'].str.contains(r'Female', na=False))
athletes.loc[mask, 'MAPPED_EVENT'] = '100m Hurdles'



mask = (athletes['EVENT'].str.contains(r'^Hurdles$', na=False) & athletes['DISTANCE'].str.contains(r'110', na=False) & athletes['DIVISION'].str.contains(r'OPEN|Open', na=False) & athletes['GENDER'].str.contains(r'Male', na=False))
athletes.loc[mask, 'MAPPED_EVENT'] = '110m Hurdles'
mask = (athletes['EVENT'].str.contains(r'^Hurdles$', na=False) & athletes['DISTANCE'].str.contains(r'110', na=False) & athletes['EVENT_CLASS'].str.contains(r'1.067', na=False) & athletes['GENDER'].str.contains(r'Male', na=False))
athletes.loc[mask, 'MAPPED_EVENT'] = '110m Hurdles'
mask = ((athletes['EVENT'].str.contains(r'110m Hurdles|110m hurdles', na=False)) 
         & ((athletes['EVENT_CLASS'].str.contains('None', na=False))|(athletes['EVENT_CLASS']==np.nan)|(athletes['EVENT_CLASS']=='')) 
         & athletes['REGION'].str.contains(r'International', na=False) & (athletes['DIVISION'].str.contains(r'None', na=False)))  # this is the correct syntax
athletes.loc[mask, 'MAPPED_EVENT'] = '110m Hurdles'

                                


mask = (athletes['EVENT'].str.contains(r'^Hurdles$', na=False) & athletes['DISTANCE'].str.contains(r'110', na=False) & athletes['EVENT_CLASS'].str.contains(r'1.067', na=False))
athletes.loc[mask, 'MAPPED_EVENT'] = '110m Hurdles'
#mask = (athletes['EVENT'].str.contains(r'^Hurdles$', na=False) & athletes['DISTANCE'].str.contains(r'110', na=False) & athletes['EVENT_CLASS'].str.contains(' ', na=True))
#athletes.loc[mask, 'MAPPED_EVENT'] = '110m Hurdles'



# Throws



mask = (athletes['EVENT'].str.contains(r'Shot Put|Shot put', na=False, regex=True) & (athletes['GENDER']=='Female') & (athletes['EVENT_CLASS']=='4kg'))# there are some additional characters after Put
athletes.loc[mask, 'MAPPED_EVENT'] = 'Shot Put'


mask = (athletes['EVENT'].str.contains(r'Shot Put|Shot put', na=False) & (athletes['GENDER']=='Male') & (athletes['EVENT_CLASS'].str.contains(r'7.26', na=False)))# there are some additional characters after Put
athletes.loc[mask, 'MAPPED_EVENT'] = 'Shot Put'
mask = (athletes['EVENT'].str.contains(r'Shot Put|Shot put', na=False) & (athletes['GENDER']=='Female') & (athletes['EVENT_CLASS'].str.contains(r'4', na=False)))# there are some additional characters after Put
athletes.loc[mask, 'MAPPED_EVENT'] = 'Shot Put'

mask = (athletes['EVENT'].str.contains(r'Shot Put|Shot put', na=False) & (athletes['DIVISION'].str.contains(r'OPEN|Open', na=False)))# there are some additional characters after Put
athletes.loc[mask, 'MAPPED_EVENT'] = 'Shot Put'

mask = (athletes['EVENT'].str.contains(r'Shot Put|Shot put', na=False) & (athletes['REGION'].str.contains(r'International', na=False)) & athletes['EVENT_CLASS'].str.contains(r'None|nan', na=False))# there are some additional characters after Put
athletes.loc[mask, 'MAPPED_EVENT'] = 'Shot Put'





# Jumps

mask = athletes['EVENT'].str.contains(r'High Jump', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = 'High Jump'

mask = athletes['EVENT'].str.contains(r'^Long\sJump$', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = 'Long Jump'
mask = athletes['EVENT'].str.contains(r'Long Jump Open', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = 'Long Jump'
mask = athletes['EVENT'].str.contains(r'Long Jump Trial', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = 'Long Jump'


mask = athletes['EVENT'].str.contains(r'Triple Jump', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = 'Triple Jump'
mask = athletes['EVENT'].str.contains(r'Pole Vault', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = 'Pole Vault'
mask = athletes['EVENT'].str.contains(r'High jump', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = 'High Jump'
mask = athletes['EVENT'].str.contains(r'Long jump', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = 'Long Jump'
mask = athletes['EVENT'].str.contains(r'Triple jump', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = 'Triple Jump'
mask = athletes['EVENT'].str.contains(r'^Pole\svault$', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = 'Pole Vault'


# Relay

mask = athletes['EVENT'].str.contains(r'4x80m Relay', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '4 x 80m'
mask = athletes['EVENT'].str.contains(r'^4\sx\s100m$', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '4 x 100m'
mask = athletes['EVENT'].str.contains(r'4x100m Relay', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '4 x 100m'
mask = athletes['EVENT'].str.contains(r'4 X 100m Relay', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '4 x 100m'
mask = (athletes['EVENT'].str.contains(r'Relay', na=False) & athletes['DISTANCE'].str.contains(r'400', na=False))
athletes.loc[mask, 'MAPPED_EVENT'] = '4 x 100m'

mask = athletes['EVENT'].str.contains(r'4x400m Relay', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '4 x 400m'
mask = athletes['EVENT'].str.contains(r'4 X 400m Relay', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '4 x 400m'
mask = athletes['EVENT'].str.contains(r'4x100 Meter Relay', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '4 x 100m'
mask = (athletes['EVENT'].str.contains(r'Relay', na=False) & athletes['DISTANCE'].str.contains(r'1600', na=False))
athletes.loc[mask, 'MAPPED_EVENT'] = '4 x 400m'
mask = athletes['EVENT'].str.contains(r'^4\sx\s400m$', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '4 x 400m'

# Decathlon/Heptathlon

mask = athletes['EVENT'].str.contains(r'^Heptathlon$', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = 'Heptathlon'
mask = athletes['EVENT'].str.contains(r'^Pentathlon$', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = 'Pentathlon'
mask = athletes['EVENT'].str.contains(r'Heptathlon', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = 'Heptathlon'
mask = athletes['EVENT'].str.contains(r'Pentathlon', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = 'Pentathlon'
'''

"\nathletes['MAPPED_EVENT']=''\n\nfor col in athletes.columns:\n    athletes[col] = athletes[col].astype(str)\n    athletes[col] = athletes[col].str.replace('\xa0', ' ', regex=True)\n    athletes[col] = athletes[col].str.replace('[\x00-\x1f\x7f-\x9f]', '', regex=True)\n    athletes[col] = athletes[col].str.replace('\r', ' ', regex=True)\n    athletes[col] = athletes[col].str.replace('\n', ' ', regex=True)\n    athletes[col] = athletes[col].str.strip()\n\n\n# Correct javelin category\n\nmask = athletes['EVENT'].str.contains(r'Javelin', na=True)\nathletes.loc[mask, 'CATEGORY_EVENT'] = 'Throw'\n\n\n# Running\n\nmask = (athletes['EVENT'].str.contains(r'Dash', na=True) & athletes['DISTANCE'].str.contains(r'60', na=True))\nathletes.loc[mask, 'MAPPED_EVENT'] = '60m'\nmask = (athletes['EVENT'].str.contains(r'Run', na=True) & athletes['DISTANCE'].str.contains(r'60', na=True))\nathletes.loc[mask, 'MAPPED_EVENT'] = '60m'\nmask = athletes['EVENT'].str.contains(r'60 Meter Run', na=True)\nathletes.l

In [26]:
# SIMPLE MAPPING RULES -  No event_class filters

def simple_map_events(athletes: pd.DataFrame) -> pd.DataFrame:
    # Columns we care about
    str_cols = ['EVENT', 'DISTANCE', 'CATEGORY_EVENT', 'DIVISION', 'GENDER', 'REGION', 'EVENT_CLASS']
    existing_cols = [c for c in str_cols if c in athletes.columns]

    # Clean text columns
    regex_cleanup = re.compile(r'[\xa0\r\n]|[\x00-\x1f\x7f-\x9f]')
    for col in existing_cols:
        athletes[col] = (
            athletes[col]
            .astype(str)
            .str.replace(regex_cleanup, ' ', regex=True)
            .str.strip()
        )

    # Initialize mapped column
    if 'MAPPED_EVENT' not in athletes:
        athletes['MAPPED_EVENT'] = np.nan

    # Correct Javelin category
    if 'EVENT' in athletes.columns and 'CATEGORY_EVENT' in athletes.columns:
        athletes.loc[athletes['EVENT'].str.contains('Javelin', na=False), 'CATEGORY_EVENT'] = 'Throw'

    # ----------------------
    # EVENT-only rules (regex on EVENT)
    # ----------------------
    event_rules = {
        r'(Dash|Run).*\b60\b|60 Meter Run|^60m$': '60m',
        r'(Dash|Run).*\b80\b|80 Meter Run|^80m$': '80m',
        r'(Dash|Run).*\b100\b|100 Meter Run|^100m$': '100m',
        r'(Dash|Run).*\b200\b|^200m$|200\sMeter': '200m',
        r'(Dash|Run).*\b300\b|^300m$|300\sMeter': '300m',
        r'(Dash|Run).*\b400\b|^400m$|400\sMeter': '400m',
        r'(Run.*800|800 Meter Run|^800m$)': '800m',
        r'Run.*1000': '1000m',
        r'(Run.*1500|^1500m$)': '1500m',
        r'(Run.*1500|^1600m$)': '1600m',
        r'(Run.*3000|^3000m$)': '3000m',
        r'(Run.*5000|^5000m$)': '5000m',
        r'(Run.*10000|^10000m$|^10,000m$)': '10,000m',
        r'Run.*Mile': '1 Mile',
    }

    for pattern, mapped in event_rules.items():
        athletes.loc[athletes['EVENT'].str.contains(pattern, na=False), 'MAPPED_EVENT'] = mapped

    # ----------------------
    # EVENT + DISTANCE rules
    # ----------------------
    distance_rules = [
        # Short sprints
        {"conditions": {"EVENT": r'(Dash|Run)', "DISTANCE": r'60'}, "map_to": "60m"},
        {"conditions": {"EVENT": r'(Dash|Run)', "DISTANCE": r'80'}, "map_to": "80m"},
        {"conditions": {"EVENT": r'(Dash|Run)', "DISTANCE": r'100'}, "map_to": "100m"},
        {"conditions": {"EVENT": r'(Dash|Run)', "DISTANCE": r'150'}, "map_to": "150m"},   
        {"conditions": {"EVENT": r'(Dash|Run)', "DISTANCE": r'200'}, "map_to": "200m"},
        {"conditions": {"EVENT": r'(Dash|Run)', "DISTANCE": r'300'}, "map_to": "300m"},
        
        {"conditions": {"EVENT": r'(Dash|Run)', "DISTANCE": r'400'}, "map_to": "400m"},
        {"conditions": {"EVENT": r'Run', "DISTANCE": r'800'}, "map_to": "800m"},
        # Middle/long
        {"conditions": {"EVENT": r'Run', "DISTANCE": r'1500'}, "map_to": "1500m"},
        {"conditions": {"EVENT": r'Run', "DISTANCE": r'1600'}, "map_to": "1600m"},
        
        {"conditions": {"EVENT": r'Run', "DISTANCE": r'3000'}, "map_to": "3000m"},
        {"conditions": {"EVENT": r'Run', "DISTANCE": r'5000'}, "map_to": "5000m"},
        {"conditions": {"EVENT": r'Run', "DISTANCE": r'10000'}, "map_to": "10,000m"},
        # Walks
        {"conditions": {"EVENT": r'1500m Racewalk'}, "map_to": "1500m Racewalk"},
        {"conditions": {"EVENT": r'3000m Racewalk'}, "map_to": "3000m Racewalk"},
        {"conditions": {"EVENT": r'5000m Race Walk'}, "map_to": "5000m Racewalk"},
        {"conditions": {"EVENT": r'Race Walk', "DISTANCE": r'10000'}, "map_to": "10000m Racewalk"},
        
        # Relays
        {"conditions": {"EVENT": r'Relay', "DISTANCE": r'400'}, "map_to": "4 x 100m"},
        {"conditions": {"EVENT": r'Relay', "DISTANCE": r'1600'}, "map_to": "4 x 400m"},
        
        # Steeple
        {"conditions": {"EVENT": r'3000m S/C'}, "map_to": "3000m Steeplechase"},
        {"conditions": {"EVENT": r'Steeplechase|S/C', "DISTANCE": r'3000'}, "map_to": "3000m Steeplechase"},
    ]

    for rule in distance_rules:
        cond = pd.Series(True, index=athletes.index)
        for col, pat in rule["conditions"].items():
            if col in athletes.columns:
                cond &= athletes[col].str.contains(pat, na=False)
            else:
                cond &= False
        athletes.loc[cond, 'MAPPED_EVENT'] = rule["map_to"]

    # ----------------------
    # Hurdles (EVENT + GENDER + EVENT_CLASS)
    # ----------------------
    hurdles_rules = [
        # 60m hurdles
        {"conditions": {"EVENT": r'(60m Hurdles|60m hurdles)'}, "map_to": "60m Hurdles"},
        {"conditions": {"EVENT": r'^Hurdles$', "DISTANCE": r'60'}, "map_to": "60m Hurdles"},

        
        # 100m hurdles
        {"conditions": {"EVENT": r'(100m Hurdles|100m hurdles)'}, "map_to": "100m Hurdles"},
        {"conditions": {"EVENT": r'^Hurdles$', "DISTANCE": r'100'}, "map_to": "100m Hurdles"},

        # 110m hurdles
        {"conditions": {"EVENT": r'(110m Hurdles|110m hurdles)'}, "map_to": "110m Hurdles"},
        {"conditions": {"EVENT": r'^Hurdles$', "DISTANCE": r'110'}, "map_to": "110m Hurdles"},

        # 200m hurdles
        {"conditions": {"EVENT": r'(200m Hurdles|200m hurdles)'}, "map_to": "200m Hurdles"},
        {"conditions": {"EVENT": r'^Hurdles$', "DISTANCE": r'200'}, "map_to": "200m Hurdles"},

        # 400m hurdles
        {"conditions": {"EVENT": r'(400m Hurdles|400m hurdles)'}, "map_to": "400m Hurdles"},
        {"conditions": {"EVENT": r'^Hurdles$', "DISTANCE": r'400'}, "map_to": "400m Hurdles"},
    ]

    for rule in hurdles_rules:
        cond = pd.Series(True, index=athletes.index)
        for col, pat in rule["conditions"].items():
            if col in athletes.columns:
                cond &= athletes[col].str.contains(pat, na=False)
            else:
                cond &= False
        athletes.loc[cond, 'MAPPED_EVENT'] = rule["map_to"]

    
    
    
 #   for rule in hurdles_rules:
 #       cond = pd.Series(True, index=athletes.index)
 #       for col, pat in rule.items():
 #           if col == "map_to":
 #               continue
 #           if col in athletes.columns:
 #               cond &= athletes[col].str.contains(pat, na=False)
 #           else:
 #               cond &= False
  #      athletes.loc[cond, 'MAPPED_EVENT'] = rule["map_to"]


    
    # ----------------------
    # Field events
    # ----------------------
    field_rules = {
        r'High Jump|High jump': 'High Jump',
        r'^Long Jump$|Long Jump Open|Long Jump Trial|Long jump|Long Jump (Zone)': 'Long Jump',
        r'Triple Jump|Triple jump': 'Triple Jump',
        r'Pole Vault|Pole vault': 'Pole Vault',
        r'Shot Put|Shot put': 'Shot Put',
        r'Javelin|Javelin Throw': 'Javelin',
        r'Discus|Discus Throw': 'Discus',
    }

    for pattern, mapped in field_rules.items():
        athletes.loc[athletes['EVENT'].str.contains(pattern, na=False), 'MAPPED_EVENT'] = mapped

    # ----------------------
    # Miscellaneous (Marathon, Heptathlon, etc.)
    # ----------------------
    misc_rules = {
        r'^Marathon$': 'Marathon',
        r'^Half Marathon$|^Half marathon$': 'Half Marathon',
        r'4x80m Relay': '4 x 80m',
        r'(4x100m Relay|4 X 100m Relay|4x100 Meter Relay|^4 x 100m$)': '4 x 100m',
        r'(4x400m Relay|4 X 400m Relay|^4 x 400m$)': '4 x 400m',
        r'Heptathlon': 'Heptathlon',
        r'Pentathlon': 'Pentathlon',
        r'Decathlon': 'Decathlon',

    }

    for pattern, mapped in misc_rules.items():
        athletes.loc[athletes['EVENT'].str.contains(pattern, na=False), 'MAPPED_EVENT'] = mapped

    return athletes


simple_map_events(athletes)


/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_49156/4184628596.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  athletes[col] = (
/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_49156/4184628596.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  athletes[col] = (
/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_49156/4184628596.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = v

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,COMPETITION,DATE,YEAR,REGION,TIMESTAMP,NOW,delta_time,delta_time_conv,event_month,MAPPED_EVENT
0,SI EN TABITHA NG,11:03.95,,,4.0,,3000m,,,,...,14TH ASEAN SCHOOLS GAMES,2025-11-23,2025,International,2025-12-09 12:38:00+00:00,2026-01-03 16:46:53.731392+00:00,41 days 16:46:53.731392,41,11,3000m
1,JE AN GARRETT CHUA,6.84,,,3.0,,Long Jump,,,,...,14TH ASEAN SCHOOLS GAMES,2025-11-23,2025,International,2025-12-09 12:38:00+00:00,2026-01-03 16:46:53.731392+00:00,41 days 16:46:53.731392,41,11,Long Jump
2,LAUREL JIA EN LIM,27.71,,,6.0,,Discus Throw,,,,...,14TH ASEAN SCHOOLS GAMES,2025-11-23,2025,International,2025-12-09 12:38:00+00:00,2026-01-03 16:46:53.731392+00:00,41 days 16:46:53.731392,41,11,Discus
3,JOSHUA SHYEN LEE,11.03,,,4.0,,100m,,,,...,14TH ASEAN SCHOOLS GAMES,2025-11-23,2025,International,2025-12-09 12:38:00+00:00,2026-01-03 16:46:53.731392+00:00,41 days 16:46:53.731392,41,11,100m
4,DARYEN XIN TZE KO,54.65,,,2.0,,400m Hurdles,,,,...,14TH ASEAN SCHOOLS GAMES,2025-11-23,2025,International,2025-12-09 12:38:00+00:00,2026-01-03 16:46:53.731392+00:00,41 days 16:46:53.731392,41,11,400m Hurdles
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
154444,"Ng, Zavier",9.88m,Hwa Chong Institution,14,18,U15,Triple Jump,0.0,None,Z854G11,...,SA Allcomers Meet 2,2025-03-02,2025,Local,2025-11-15 13:47:37.773700,2026-01-03 16:46:53.731392+00:00,307 days 16:46:53.731392,307,3,Triple Jump
154445,"Choo Jia Yi, Allyson",9.30m,Team Start Singapore,17.0,3,Open,Triple Jump,0.0,None,A967D08,...,SA Allcomers Meet 3,2025-08-30,2025,Local,2025-11-15 13:49:23.923169,2026-01-03 16:46:53.731392+00:00,126 days 16:46:53.731392,126,8,Triple Jump
154450,Lua Yu Xuan,10.02,NYGH,,1.0,C,Triple Jump,,,,...,National School Games,2025-04-18,2025,Local,2025-11-17 17:34:00+00:00,2026-01-03 16:46:53.731392+00:00,260 days 16:46:53.731392,260,4,Triple Jump
154451,Muhammad Aaryan Shah Bin Azhar,12.61,SSP,,3.0,B,Triple Jump,,,,...,National School Games,2025-04-13,2025,Local,2025-11-17 17:34:00+00:00,2026-01-03 16:46:53.731392+00:00,265 days 16:46:53.731392,265,4,Triple Jump


In [27]:
athletes[athletes['MAPPED_EVENT']=='']

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,COMPETITION,DATE,YEAR,REGION,TIMESTAMP,NOW,delta_time,delta_time_conv,event_month,MAPPED_EVENT


In [28]:
for col in athletes.columns:
    athletes[col] = athletes[col].astype(str)
    athletes[col] = athletes[col].str.replace('\xa0', ' ', regex=True)
    athletes[col] = athletes[col].str.replace('[\x00-\x1f\x7f-\x9f]', '', regex=True)
    athletes[col] = athletes[col].str.replace('\r', ' ', regex=True)
    athletes[col] = athletes[col].str.replace('\n', ' ', regex=True)
    athletes[col] = athletes[col].str.strip()

/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_49156/2203170124.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  athletes[col] = athletes[col].astype(str)
/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_49156/2203170124.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  athletes[col] = athletes[col].str.replace('\xa0', ' ', regex=True)
/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_49156/2203170124.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of

/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_81269/2203170124.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  athletes[col] = athletes[col].str.replace('\xa0', ' ', regex=True)
/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_81269/2203170124.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  athletes[col] = athletes[col].str.replace('[\x00-\x1f\x7f-\x9f]', '', regex=True)
/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_81269/2203170124.py:5: SettingWithCopyWarning: 


/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_81269/2203170124.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  athletes[col] = athletes[col].str.strip()
/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_81269/2203170124.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  athletes[col] = athletes[col].astype(str)
/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_81269/2203170124.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

In [29]:
athletes[athletes['MAPPED_EVENT']!='']

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,COMPETITION,DATE,YEAR,REGION,TIMESTAMP,NOW,delta_time,delta_time_conv,event_month,MAPPED_EVENT
0,SI EN TABITHA NG,11:03.95,,,4.0,,3000m,,,,...,14TH ASEAN SCHOOLS GAMES,2025-11-23 00:00:00,2025,International,2025-12-09 12:38:00+00:00,2026-01-03 16:46:53.731392+00:00,41 days 16:46:53.731392,41,11,3000m
1,JE AN GARRETT CHUA,6.84,,,3.0,,Long Jump,,,,...,14TH ASEAN SCHOOLS GAMES,2025-11-23 00:00:00,2025,International,2025-12-09 12:38:00+00:00,2026-01-03 16:46:53.731392+00:00,41 days 16:46:53.731392,41,11,Long Jump
2,LAUREL JIA EN LIM,27.71,,,6.0,,Discus Throw,,,,...,14TH ASEAN SCHOOLS GAMES,2025-11-23 00:00:00,2025,International,2025-12-09 12:38:00+00:00,2026-01-03 16:46:53.731392+00:00,41 days 16:46:53.731392,41,11,Discus
3,JOSHUA SHYEN LEE,11.03,,,4.0,,100m,,,,...,14TH ASEAN SCHOOLS GAMES,2025-11-23 00:00:00,2025,International,2025-12-09 12:38:00+00:00,2026-01-03 16:46:53.731392+00:00,41 days 16:46:53.731392,41,11,100m
4,DARYEN XIN TZE KO,54.65,,,2.0,,400m Hurdles,,,,...,14TH ASEAN SCHOOLS GAMES,2025-11-23 00:00:00,2025,International,2025-12-09 12:38:00+00:00,2026-01-03 16:46:53.731392+00:00,41 days 16:46:53.731392,41,11,400m Hurdles
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
154444,"Ng, Zavier",9.88m,Hwa Chong Institution,14,18,U15,Triple Jump,0.0,None,Z854G11,...,SA Allcomers Meet 2,2025-03-02 00:00:00,2025,Local,2025-11-15 13:47:37.773700,2026-01-03 16:46:53.731392+00:00,307 days 16:46:53.731392,307,3,Triple Jump
154445,"Choo Jia Yi, Allyson",9.30m,Team Start Singapore,17.0,3,Open,Triple Jump,0.0,None,A967D08,...,SA Allcomers Meet 3,2025-08-30 00:00:00,2025,Local,2025-11-15 13:49:23.923169,2026-01-03 16:46:53.731392+00:00,126 days 16:46:53.731392,126,8,Triple Jump
154450,Lua Yu Xuan,10.02,NYGH,,1.0,C,Triple Jump,,,,...,National School Games,2025-04-18 00:00:00,2025,Local,2025-11-17 17:34:00+00:00,2026-01-03 16:46:53.731392+00:00,260 days 16:46:53.731392,260,4,Triple Jump
154451,Muhammad Aaryan Shah Bin Azhar,12.61,SSP,,3.0,B,Triple Jump,,,,...,National School Games,2025-04-13 00:00:00,2025,Local,2025-11-17 17:34:00+00:00,2026-01-03 16:46:53.731392+00:00,265 days 16:46:53.731392,265,4,Triple Jump


In [30]:
athletes

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,COMPETITION,DATE,YEAR,REGION,TIMESTAMP,NOW,delta_time,delta_time_conv,event_month,MAPPED_EVENT
0,SI EN TABITHA NG,11:03.95,,,4.0,,3000m,,,,...,14TH ASEAN SCHOOLS GAMES,2025-11-23 00:00:00,2025,International,2025-12-09 12:38:00+00:00,2026-01-03 16:46:53.731392+00:00,41 days 16:46:53.731392,41,11,3000m
1,JE AN GARRETT CHUA,6.84,,,3.0,,Long Jump,,,,...,14TH ASEAN SCHOOLS GAMES,2025-11-23 00:00:00,2025,International,2025-12-09 12:38:00+00:00,2026-01-03 16:46:53.731392+00:00,41 days 16:46:53.731392,41,11,Long Jump
2,LAUREL JIA EN LIM,27.71,,,6.0,,Discus Throw,,,,...,14TH ASEAN SCHOOLS GAMES,2025-11-23 00:00:00,2025,International,2025-12-09 12:38:00+00:00,2026-01-03 16:46:53.731392+00:00,41 days 16:46:53.731392,41,11,Discus
3,JOSHUA SHYEN LEE,11.03,,,4.0,,100m,,,,...,14TH ASEAN SCHOOLS GAMES,2025-11-23 00:00:00,2025,International,2025-12-09 12:38:00+00:00,2026-01-03 16:46:53.731392+00:00,41 days 16:46:53.731392,41,11,100m
4,DARYEN XIN TZE KO,54.65,,,2.0,,400m Hurdles,,,,...,14TH ASEAN SCHOOLS GAMES,2025-11-23 00:00:00,2025,International,2025-12-09 12:38:00+00:00,2026-01-03 16:46:53.731392+00:00,41 days 16:46:53.731392,41,11,400m Hurdles
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
154444,"Ng, Zavier",9.88m,Hwa Chong Institution,14,18,U15,Triple Jump,0.0,None,Z854G11,...,SA Allcomers Meet 2,2025-03-02 00:00:00,2025,Local,2025-11-15 13:47:37.773700,2026-01-03 16:46:53.731392+00:00,307 days 16:46:53.731392,307,3,Triple Jump
154445,"Choo Jia Yi, Allyson",9.30m,Team Start Singapore,17.0,3,Open,Triple Jump,0.0,None,A967D08,...,SA Allcomers Meet 3,2025-08-30 00:00:00,2025,Local,2025-11-15 13:49:23.923169,2026-01-03 16:46:53.731392+00:00,126 days 16:46:53.731392,126,8,Triple Jump
154450,Lua Yu Xuan,10.02,NYGH,,1.0,C,Triple Jump,,,,...,National School Games,2025-04-18 00:00:00,2025,Local,2025-11-17 17:34:00+00:00,2026-01-03 16:46:53.731392+00:00,260 days 16:46:53.731392,260,4,Triple Jump
154451,Muhammad Aaryan Shah Bin Azhar,12.61,SSP,,3.0,B,Triple Jump,,,,...,National School Games,2025-04-13 00:00:00,2025,Local,2025-11-17 17:34:00+00:00,2026-01-03 16:46:53.731392+00:00,265 days 16:46:53.731392,265,4,Triple Jump


In [31]:
os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/Asian Indoor/')


athletes.to_csv('athletes_post_map_jan3.csv', sep=',', encoding='utf-8-sig', index=False)


# Load Benchmarks

In [32]:
# Load Benchmarks

os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/Benchmarks/')

benchmarks = pd.read_csv("2026 Asian Indoor.csv")


In [33]:
benchmarks

,COMPETITION,EVENT,GENDER,INDOOR,OUTDOOR,BENCHMARK
0,2026 Asian Indoor,60m,Male,6.7,NaN,6.7
1,2026 Asian Indoor,60m,Female,7.46,NaN,7.46
2,2026 Asian Indoor,100m,Male,NaN,10.32,10.32
3,2026 Asian Indoor,100m,Female,NaN,11.64,11.64
4,2026 Asian Indoor,400m,Male,49.23,48.25,48.25
5,2026 Asian Indoor,400m,Female,56.25,55.3,55.3
6,2026 Asian Indoor,800m,Male,01:53.61,01:51.71,01:51.71
7,2026 Asian Indoor,800m,Female,02:18.89,02:15.94,02:15.94
8,2026 Asian Indoor,1500m,Male,03:51.59,03:48.09,03:48.09
9,2026 Asian Indoor,1500m,Female,04:35.99,04:33.59,04:33.59


In [34]:
benchmarks = benchmarks[['COMPETITION', 'EVENT', 'GENDER', 'BENCHMARK']]

In [35]:
benchmarks

,COMPETITION,EVENT,GENDER,BENCHMARK
0,2026 Asian Indoor,60m,Male,6.7
1,2026 Asian Indoor,60m,Female,7.46
2,2026 Asian Indoor,100m,Male,10.32
3,2026 Asian Indoor,100m,Female,11.64
4,2026 Asian Indoor,400m,Male,48.25
5,2026 Asian Indoor,400m,Female,55.3
6,2026 Asian Indoor,800m,Male,01:51.71
7,2026 Asian Indoor,800m,Female,02:15.94
8,2026 Asian Indoor,1500m,Male,03:48.09
9,2026 Asian Indoor,1500m,Female,04:33.59


In [36]:
def convert_time_refactored(i, string, metric):
    """
    Convert various metric formats (distance, time) to a float value (primarily seconds for times).
    Optimized for speed: no global variables, no print statements, no unnecessary conversions.

    Args:
        i (int): Index (unused, kept for compatibility).
        string (str): Event description.
        metric (str, float, or datetime): The result metric.

    Returns:
        float or empty string: Converted metric as float (seconds/meters), or '' if not convertible.
    """
    l = ['discus', 'throw', 'jump', 'vault', 'shot']
    string = str(string).lower()
    metric_str = str(metric)
    output = ''

    try:
        # Skip marks with illegal wind speeds
        if isinstance(metric_str, str) and 'w' in metric_str.lower():
            return ''

        # Field events (distances)
        if any(s in string for s in l):
            # Remove unit if present
            metric_clean = metric_str.replace('m', '').replace('GR', '')
            return float(metric_clean)

        # No event description
        if string == '':
            return ''

        # Time events
        count_colon = metric_str.count(':')
        count_dot = metric_str.count('.')

        # Simple time as float (no colon)
        if count_colon == 0:
            return float(metric_str)

        # Convert time formats with two colons (like XX:XX:XX, XX:XX.XX)
        if count_colon == 2:
            # For 10,000m and 5000m, replace the 6th character with '.' for format XX:XX.XX
            if ('10,000m' in string or '5000m' in string or '1500m' in string):
                if len(metric_str) == 7:  # X:XX:XX (1500m special case)
                    idx = 4
                    metric_mod = '0' + metric_str[:idx] + '.' + metric_str[idx+1:]
                else:
                    idx = 5
                    metric_mod = metric_str[:idx] + '.' + metric_str[idx+1:]
                m, s = metric_mod.split(':')[-2:]
                return float(
                    (int(m) * 60) + float(s)
                )

            # Standard HH:MM:SS
            h, m, s = metric_str.split(':')
            return float(
                int(h) * 3600 + int(m) * 60 + float(s)
            )

        # Handle datetime.time/datetime.datetime objects
        if isinstance(metric, (datetime.time, datetime.datetime)):
            t = str(metric)
            h, m, s = t.split(':')
            return float(int(h) * 3600 + int(m) * 60 + float(s))

        # MM:SS.sss format
        if count_colon == 1 and count_dot >= 1:
            m, s = metric_str.split(':')
            return float(int(m) * 60 + float(s))

        # HH.MM.SS (rare) or MM:SS:SS
        if count_colon == 1 and count_dot == 2:
            # Replace first dot with colon
            metric_mod = metric_str.replace('.', ':', 1)
            h, m, s = metric_mod.split(':')
            return float(int(h) * 3600 + int(m) * 60 + float(s))

        # HH:MM:SS (no dots)
        if count_colon == 2 and count_dot == 0:
            h, m, s = metric_str.split(':')
            return float(int(h) * 3600 + int(m) * 60 + float(s))

        # MM:SS (no dots)
        if count_colon == 1 and count_dot == 0:
            m, s = metric_str.split(':')
            return float(int(m) * 60 + int(s))

    except Exception:
        return ''

    return output

In [37]:
def process_benchmarks(df):
    
    for i in range(len(df)):

        rowIndex = df.index[i]

        input_string=df.iloc[rowIndex,1]
    
    #    indoor = df.iloc[rowIndex,3]
        outdoor = df.iloc[rowIndex,3]
    
        
    #    if indoor==None or outdoor==None:
        if outdoor==None:
    
            
            continue
        
    #    indoor_out = convert_time_refactored(i, input_string, indoor)
        outdoor_out = convert_time_refactored(i, input_string, outdoor)

        
    
     #   df.loc[rowIndex, 'INDOOR_C'] = indoor_out
        df.loc[rowIndex, 'BENCHMARK_C'] = outdoor_out

    return df

In [38]:
process_benchmarks(benchmarks)

/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_49156/558968220.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.loc[rowIndex, 'BENCHMARK_C'] = outdoor_out


,COMPETITION,EVENT,GENDER,BENCHMARK,BENCHMARK_C
0,2026 Asian Indoor,60m,Male,6.7,6.70
1,2026 Asian Indoor,60m,Female,7.46,7.46
2,2026 Asian Indoor,100m,Male,10.32,10.32
3,2026 Asian Indoor,100m,Female,11.64,11.64
4,2026 Asian Indoor,400m,Male,48.25,48.25
5,2026 Asian Indoor,400m,Female,55.3,55.30
6,2026 Asian Indoor,800m,Male,01:51.71,111.71
7,2026 Asian Indoor,800m,Female,02:15.94,135.94
8,2026 Asian Indoor,1500m,Male,03:48.09,228.09
9,2026 Asian Indoor,1500m,Female,04:33.59,273.59


In [39]:
'''
mask = benchmarks['EVENT'].str.lower().str.contains(r'jump|throw|put|pole|decathlon|heptathlon', na=True)

benchmarks.loc[mask, '1%']=benchmarks['INDOOR_C']*0.99
benchmarks.loc[mask, '2%']=benchmarks['INDOOR_C']*0.98
benchmarks.loc[mask, '3.5%']=benchmarks['INDOOR_C']*0.965
benchmarks.loc[mask, '5%']=benchmarks['INDOOR_C']*0.95

benchmarks.loc[~mask, '1%']=benchmarks['INDOOR_C']*1.01
benchmarks.loc[~mask, '2%']=benchmarks['INDOOR_C']*1.02
benchmarks.loc[~mask, '3.5%']=benchmarks['INDOOR_C']*1.035
benchmarks.loc[~mask, '5%']=benchmarks['INDOOR_C']*1.05

benchmarks.loc[mask, '1%(o)']=benchmarks['OUTDOOR_C']*0.99
benchmarks.loc[mask, '2%(o)']=benchmarks['OUTDOOR_C']*0.98
benchmarks.loc[mask, '3.5%(o)']=benchmarks['OUTDOOR_C']*0.965
benchmarks.loc[mask, '5%(o)']=benchmarks['OUTDOOR_C']*0.95

benchmarks.loc[~mask, '1%(o)']=benchmarks['OUTDOOR_C']*1.01
benchmarks.loc[~mask, '2%(o)']=benchmarks['OUTDOOR_C']*1.02
benchmarks.loc[~mask, '3.5%(o)']=benchmarks['OUTDOOR_C']*1.035
benchmarks.loc[~mask, '5%(o)']=benchmarks['OUTDOOR_C']*1.05

'''

"\nmask = benchmarks['EVENT'].str.lower().str.contains(r'jump|throw|put|pole|decathlon|heptathlon', na=True)\n\nbenchmarks.loc[mask, '1%']=benchmarks['INDOOR_C']*0.99\nbenchmarks.loc[mask, '2%']=benchmarks['INDOOR_C']*0.98\nbenchmarks.loc[mask, '3.5%']=benchmarks['INDOOR_C']*0.965\nbenchmarks.loc[mask, '5%']=benchmarks['INDOOR_C']*0.95\n\nbenchmarks.loc[~mask, '1%']=benchmarks['INDOOR_C']*1.01\nbenchmarks.loc[~mask, '2%']=benchmarks['INDOOR_C']*1.02\nbenchmarks.loc[~mask, '3.5%']=benchmarks['INDOOR_C']*1.035\nbenchmarks.loc[~mask, '5%']=benchmarks['INDOOR_C']*1.05\n\nbenchmarks.loc[mask, '1%(o)']=benchmarks['OUTDOOR_C']*0.99\nbenchmarks.loc[mask, '2%(o)']=benchmarks['OUTDOOR_C']*0.98\nbenchmarks.loc[mask, '3.5%(o)']=benchmarks['OUTDOOR_C']*0.965\nbenchmarks.loc[mask, '5%(o)']=benchmarks['OUTDOOR_C']*0.95\n\nbenchmarks.loc[~mask, '1%(o)']=benchmarks['OUTDOOR_C']*1.01\nbenchmarks.loc[~mask, '2%(o)']=benchmarks['OUTDOOR_C']*1.02\nbenchmarks.loc[~mask, '3.5%(o)']=benchmarks['OUTDOOR_C']*1.

In [40]:
mask = benchmarks['EVENT'].str.lower().str.contains(r'jump|throw|put|pole|decathlon|heptathlon', na=True)

benchmarks.loc[mask, '1%']=benchmarks['BENCHMARK_C']*0.99
benchmarks.loc[mask, '2%']=benchmarks['BENCHMARK_C']*0.98
benchmarks.loc[mask, '3.5%']=benchmarks['BENCHMARK_C']*0.965
benchmarks.loc[mask, '5%']=benchmarks['BENCHMARK_C']*0.95

benchmarks.loc[~mask, '1%']=benchmarks['BENCHMARK_C']*1.01
benchmarks.loc[~mask, '2%']=benchmarks['BENCHMARK_C']*1.02
benchmarks.loc[~mask, '3.5%']=benchmarks['BENCHMARK_C']*1.035
benchmarks.loc[~mask, '5%']=benchmarks['BENCHMARK_C']*1.05



In [41]:
benchmarks['MAPPED_EVENT']=benchmarks['EVENT'].str.strip()

In [42]:
for col in benchmarks.columns:
    benchmarks[col] = benchmarks[col].astype(str)
    benchmarks[col] = benchmarks[col].str.replace('\xa0', ' ', regex=True)
    benchmarks[col] = benchmarks[col].str.replace('[\x00-\x1f\x7f-\x9f]', '', regex=True)
    benchmarks[col] = benchmarks[col].str.replace('\r', ' ', regex=True)
    benchmarks[col] = benchmarks[col].str.replace('\n', ' ', regex=True)
    benchmarks[col] = benchmarks[col].str.strip()


In [43]:
benchmarks

,COMPETITION,EVENT,GENDER,BENCHMARK,BENCHMARK_C,1%,2%,3.5%,5%,MAPPED_EVENT
0,2026 Asian Indoor,60m,Male,6.7,6.7,6.767,6.8340000000000005,6.9345,7.035,60m
1,2026 Asian Indoor,60m,Female,7.46,7.46,7.5346,7.6092,7.721099999999999,7.833,60m
2,2026 Asian Indoor,100m,Male,10.32,10.32,10.4232,10.5264,10.681199999999999,10.836,100m
3,2026 Asian Indoor,100m,Female,11.64,11.64,11.756400000000001,11.872800000000002,12.0474,12.222000000000001,100m
4,2026 Asian Indoor,400m,Male,48.25,48.25,48.7325,49.215,49.93875,50.6625,400m
5,2026 Asian Indoor,400m,Female,55.3,55.3,55.852999999999994,56.406,57.235499999999995,58.065,400m
6,2026 Asian Indoor,800m,Male,01:51.71,111.71000000000001,112.82710000000002,113.94420000000001,115.61985,117.29550000000002,800m
7,2026 Asian Indoor,800m,Female,02:15.94,135.94,137.2994,138.6588,140.69789999999998,142.737,800m
8,2026 Asian Indoor,1500m,Male,03:48.09,228.09,230.3709,232.6518,236.07315,239.49450000000002,1500m
9,2026 Asian Indoor,1500m,Female,04:33.59,273.59000000000003,276.32590000000005,279.06180000000006,283.16565,287.26950000000005,1500m


In [44]:
athletes

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,COMPETITION,DATE,YEAR,REGION,TIMESTAMP,NOW,delta_time,delta_time_conv,event_month,MAPPED_EVENT
0,SI EN TABITHA NG,11:03.95,,,4.0,,3000m,,,,...,14TH ASEAN SCHOOLS GAMES,2025-11-23 00:00:00,2025,International,2025-12-09 12:38:00+00:00,2026-01-03 16:46:53.731392+00:00,41 days 16:46:53.731392,41,11,3000m
1,JE AN GARRETT CHUA,6.84,,,3.0,,Long Jump,,,,...,14TH ASEAN SCHOOLS GAMES,2025-11-23 00:00:00,2025,International,2025-12-09 12:38:00+00:00,2026-01-03 16:46:53.731392+00:00,41 days 16:46:53.731392,41,11,Long Jump
2,LAUREL JIA EN LIM,27.71,,,6.0,,Discus Throw,,,,...,14TH ASEAN SCHOOLS GAMES,2025-11-23 00:00:00,2025,International,2025-12-09 12:38:00+00:00,2026-01-03 16:46:53.731392+00:00,41 days 16:46:53.731392,41,11,Discus
3,JOSHUA SHYEN LEE,11.03,,,4.0,,100m,,,,...,14TH ASEAN SCHOOLS GAMES,2025-11-23 00:00:00,2025,International,2025-12-09 12:38:00+00:00,2026-01-03 16:46:53.731392+00:00,41 days 16:46:53.731392,41,11,100m
4,DARYEN XIN TZE KO,54.65,,,2.0,,400m Hurdles,,,,...,14TH ASEAN SCHOOLS GAMES,2025-11-23 00:00:00,2025,International,2025-12-09 12:38:00+00:00,2026-01-03 16:46:53.731392+00:00,41 days 16:46:53.731392,41,11,400m Hurdles
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
154444,"Ng, Zavier",9.88m,Hwa Chong Institution,14,18,U15,Triple Jump,0.0,None,Z854G11,...,SA Allcomers Meet 2,2025-03-02 00:00:00,2025,Local,2025-11-15 13:47:37.773700,2026-01-03 16:46:53.731392+00:00,307 days 16:46:53.731392,307,3,Triple Jump
154445,"Choo Jia Yi, Allyson",9.30m,Team Start Singapore,17.0,3,Open,Triple Jump,0.0,None,A967D08,...,SA Allcomers Meet 3,2025-08-30 00:00:00,2025,Local,2025-11-15 13:49:23.923169,2026-01-03 16:46:53.731392+00:00,126 days 16:46:53.731392,126,8,Triple Jump
154450,Lua Yu Xuan,10.02,NYGH,,1.0,C,Triple Jump,,,,...,National School Games,2025-04-18 00:00:00,2025,Local,2025-11-17 17:34:00+00:00,2026-01-03 16:46:53.731392+00:00,260 days 16:46:53.731392,260,4,Triple Jump
154451,Muhammad Aaryan Shah Bin Azhar,12.61,SSP,,3.0,B,Triple Jump,,,,...,National School Games,2025-04-13 00:00:00,2025,Local,2025-11-17 17:34:00+00:00,2026-01-03 16:46:53.731392+00:00,265 days 16:46:53.731392,265,4,Triple Jump


In [45]:
# Merge benchmarks onto athletes on MAPPED_EVENT and GENDER

df = pd.merge(
    left=athletes, 
    right=benchmarks,
    how='left',
    left_on=['MAPPED_EVENT', 'GENDER'],
    right_on=['MAPPED_EVENT', 'GENDER'],
)

In [46]:
df

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT_x,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,event_month,MAPPED_EVENT,COMPETITION_y,EVENT_y,BENCHMARK,BENCHMARK_C,1%,2%,3.5%,5%
0,SI EN TABITHA NG,11:03.95,,,4.0,,3000m,,,,...,11,3000m,2026 Asian Indoor,3000m,09:55.09,595.09,601.0409000000001,606.9918,615.91815,624.8445
1,JE AN GARRETT CHUA,6.84,,,3.0,,Long Jump,,,,...,11,Long Jump,2026 Asian Indoor,Long Jump,7.61,7.61,7.5339,7.4578,7.34365,7.2295
2,LAUREL JIA EN LIM,27.71,,,6.0,,Discus Throw,,,,...,11,Discus,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,JOSHUA SHYEN LEE,11.03,,,4.0,,100m,,,,...,11,100m,2026 Asian Indoor,100m,10.32,10.32,10.4232,10.5264,10.681199999999999,10.836
4,DARYEN XIN TZE KO,54.65,,,2.0,,400m Hurdles,,,,...,11,400m Hurdles,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20241,"Ng, Zavier",9.88m,Hwa Chong Institution,14,18,U15,Triple Jump,0.0,None,Z854G11,...,3,Triple Jump,2026 Asian Indoor,Triple Jump,15.96,15.96,15.800400000000002,15.6408,15.4014,15.162
20242,"Choo Jia Yi, Allyson",9.30m,Team Start Singapore,17.0,3,Open,Triple Jump,0.0,None,A967D08,...,8,Triple Jump,2026 Asian Indoor,Triple Jump,12.65,12.65,12.5235,12.397,12.20725,12.0175
20243,Lua Yu Xuan,10.02,NYGH,,1.0,C,Triple Jump,,,,...,4,Triple Jump,2026 Asian Indoor,Triple Jump,12.65,12.65,12.5235,12.397,12.20725,12.0175
20244,Muhammad Aaryan Shah Bin Azhar,12.61,SSP,,3.0,B,Triple Jump,,,,...,4,Triple Jump,2026 Asian Indoor,Triple Jump,15.96,15.96,15.800400000000002,15.6408,15.4014,15.162


In [47]:
# Identify rows causing left join to produce larger dataframe

duplicates_in_right = benchmarks[benchmarks.duplicated(subset=['MAPPED_EVENT', 'GENDER'], keep=False)]


In [48]:
duplicates_in_right

,COMPETITION,EVENT,GENDER,BENCHMARK,BENCHMARK_C,1%,2%,3.5%,5%,MAPPED_EVENT


In [49]:
#left_df_with_duplicates = athletes[athletes['MAPPED_EVENT', 'GENDER'].isin(duplicates_in_right['MAPPED_EVENT', 'GENDER'])]

In [50]:
# replace '-' with NaN

df['RESULT'] = df['RESULT'].replace(regex=r'–', value=np.NaN)
#df['SEED'] = df['SEED'].replace(regex=r'–', value=np.NaN)


In [51]:
df

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT_x,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,event_month,MAPPED_EVENT,COMPETITION_y,EVENT_y,BENCHMARK,BENCHMARK_C,1%,2%,3.5%,5%
0,SI EN TABITHA NG,11:03.95,,,4.0,,3000m,,,,...,11,3000m,2026 Asian Indoor,3000m,09:55.09,595.09,601.0409000000001,606.9918,615.91815,624.8445
1,JE AN GARRETT CHUA,6.84,,,3.0,,Long Jump,,,,...,11,Long Jump,2026 Asian Indoor,Long Jump,7.61,7.61,7.5339,7.4578,7.34365,7.2295
2,LAUREL JIA EN LIM,27.71,,,6.0,,Discus Throw,,,,...,11,Discus,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,JOSHUA SHYEN LEE,11.03,,,4.0,,100m,,,,...,11,100m,2026 Asian Indoor,100m,10.32,10.32,10.4232,10.5264,10.681199999999999,10.836
4,DARYEN XIN TZE KO,54.65,,,2.0,,400m Hurdles,,,,...,11,400m Hurdles,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20241,"Ng, Zavier",9.88m,Hwa Chong Institution,14,18,U15,Triple Jump,0.0,None,Z854G11,...,3,Triple Jump,2026 Asian Indoor,Triple Jump,15.96,15.96,15.800400000000002,15.6408,15.4014,15.162
20242,"Choo Jia Yi, Allyson",9.30m,Team Start Singapore,17.0,3,Open,Triple Jump,0.0,None,A967D08,...,8,Triple Jump,2026 Asian Indoor,Triple Jump,12.65,12.65,12.5235,12.397,12.20725,12.0175
20243,Lua Yu Xuan,10.02,NYGH,,1.0,C,Triple Jump,,,,...,4,Triple Jump,2026 Asian Indoor,Triple Jump,12.65,12.65,12.5235,12.397,12.20725,12.0175
20244,Muhammad Aaryan Shah Bin Azhar,12.61,SSP,,3.0,B,Triple Jump,,,,...,4,Triple Jump,2026 Asian Indoor,Triple Jump,15.96,15.96,15.800400000000002,15.6408,15.4014,15.162


In [52]:
os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/Asian Indoor/')


df.to_csv('asian_indoor_postmap_jan3.csv', sep=',', encoding='utf-8-sig', index=False)


In [53]:
# Convert results and seed into seconds format for mapped events only (vectorised version)

# First, normalize data as you did (remove control chars etc.)
for col in df.columns:
    df[col] = df[col].astype(str)
    df[col] = df[col].str.replace('\xa0', ' ', regex=True)
    df[col] = df[col].str.replace('[\x00-\x1f\x7f-\x9f]', '', regex=True)
    df[col] = df[col].str.replace('\r', ' ', regex=True)
    df[col] = df[col].str.replace('\n', ' ', regex=True)
    df[col] = df[col].str.strip()

# Define a filter for rows with convertible results
invalid_results = {'—', 'None', 'DQ', 'SCR', 'FS', 'DNQ', 'DNS', 'NH', 'NM', 'FOUL', 'DNF', 'SR'}

# Apply conversion vectorized using apply, skipping invalid values
def convert_for_row(row):
    if row['RESULT'] in invalid_results:
        return ''
    return convert_time_refactored(row.name, row['MAPPED_EVENT'], row['RESULT'])

df['RESULT_CONV'] = df.apply(convert_for_row, axis=1)


In [54]:
df

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT_x,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,MAPPED_EVENT,COMPETITION_y,EVENT_y,BENCHMARK,BENCHMARK_C,1%,2%,3.5%,5%,RESULT_CONV
0,SI EN TABITHA NG,11:03.95,,,4.0,,3000m,,,,...,3000m,2026 Asian Indoor,3000m,09:55.09,595.09,601.0409000000001,606.9918,615.91815,624.8445,663.95
1,JE AN GARRETT CHUA,6.84,,,3.0,,Long Jump,,,,...,Long Jump,2026 Asian Indoor,Long Jump,7.61,7.61,7.5339,7.4578,7.34365,7.2295,6.84
2,LAUREL JIA EN LIM,27.71,,,6.0,,Discus Throw,,,,...,Discus,nan,nan,nan,nan,nan,nan,nan,nan,27.71
3,JOSHUA SHYEN LEE,11.03,,,4.0,,100m,,,,...,100m,2026 Asian Indoor,100m,10.32,10.32,10.4232,10.5264,10.681199999999999,10.836,11.03
4,DARYEN XIN TZE KO,54.65,,,2.0,,400m Hurdles,,,,...,400m Hurdles,nan,nan,nan,nan,nan,nan,nan,nan,54.65
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20241,"Ng, Zavier",9.88m,Hwa Chong Institution,14,18,U15,Triple Jump,0.0,None,Z854G11,...,Triple Jump,2026 Asian Indoor,Triple Jump,15.96,15.96,15.800400000000002,15.6408,15.4014,15.162,9.88
20242,"Choo Jia Yi, Allyson",9.30m,Team Start Singapore,17.0,3,Open,Triple Jump,0.0,None,A967D08,...,Triple Jump,2026 Asian Indoor,Triple Jump,12.65,12.65,12.5235,12.397,12.20725,12.0175,9.3
20243,Lua Yu Xuan,10.02,NYGH,,1.0,C,Triple Jump,,,,...,Triple Jump,2026 Asian Indoor,Triple Jump,12.65,12.65,12.5235,12.397,12.20725,12.0175,10.02
20244,Muhammad Aaryan Shah Bin Azhar,12.61,SSP,,3.0,B,Triple Jump,,,,...,Triple Jump,2026 Asian Indoor,Triple Jump,15.96,15.96,15.800400000000002,15.6408,15.4014,15.162,12.61


In [55]:
# Choose SEED if better than RESULT

df['RESULT_BEST'] = df['RESULT_CONV']

In [56]:
df.columns

Index(['NAME', 'RESULT', 'TEAM', 'AGE', 'COMPETITION_RANK', 'DIVISION',
       'EVENT_x', 'DISTANCE', 'EVENT_CLASS', 'UNIQUE_ID', 'DOB', 'NATIONALITY',
       'WIND', 'CATEGORY_EVENT', 'GENDER', 'COMPETITION_x', 'DATE', 'YEAR',
       'REGION', 'TIMESTAMP', 'NOW', 'delta_time', 'delta_time_conv',
       'event_month', 'MAPPED_EVENT', 'COMPETITION_y', 'EVENT_y', 'BENCHMARK',
       'BENCHMARK_C', '1%', '2%', '3.5%', '5%', 'RESULT_CONV', 'RESULT_BEST'],
      dtype='object')

In [57]:
# Change to numeric

df[['1%', '2%', '3.5%', '5%','RESULT_BEST', 'BENCHMARK_C']] = df[['1%', '2%', '3.5%', '5%', 'RESULT_BEST', 'BENCHMARK_C']].apply(pd.to_numeric, errors='coerce')

In [58]:
mask = df['CATEGORY_EVENT'].str.lower().str.contains(r'jump|throw|decathlon|heptathlon', na=True)

df.loc[mask, 'Delta1'] = df['RESULT_BEST']-df['1%']
df.loc[mask, 'Delta2'] = df['RESULT_BEST']-df['2%']
df.loc[mask, 'Delta3.5'] = df['RESULT_BEST']-df['3.5%']
df.loc[mask, 'Delta5'] = df['RESULT_BEST']-df['5%']
df.loc[mask, 'Delta_Benchmark'] = df['RESULT_BEST']-df['BENCHMARK_C']

df.loc[~mask, 'Delta1'] =  df['1%'] - df['RESULT_BEST']
df.loc[~mask, 'Delta2'] =  df['2%'] - df['RESULT_BEST']
df.loc[~mask, 'Delta3.5'] = df['3.5%'] - df['RESULT_BEST']
df.loc[~mask, 'Delta5'] = df['5%'] - df['RESULT_BEST']
df.loc[~mask, 'Delta_Benchmark'] = df['BENCHMARK_C'] - df['RESULT_BEST']

#df=df.loc[df['COMPETITION']!='Southeast Asian Games'] # Do not include results from SEAG in dataset

In [59]:
# Performance metric to filter out athletes

df['PERF_SCALAR']=df['Delta5']/df['BENCHMARK_C']*100

In [60]:
os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/Asian Indoor/')


df.to_csv('asian_indoor_postmap_benchmarked_jan3.csv', sep=',', encoding='utf-8-sig', index=False)


In [61]:
df

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT_x,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,3.5%,5%,RESULT_CONV,RESULT_BEST,Delta1,Delta2,Delta3.5,Delta5,Delta_Benchmark,PERF_SCALAR
0,SI EN TABITHA NG,11:03.95,,,4.0,,3000m,,,,...,615.91815,624.8445,663.95,663.95,-62.9091,-56.9582,-48.03185,-39.1055,-68.86,-6.571359
1,JE AN GARRETT CHUA,6.84,,,3.0,,Long Jump,,,,...,7.34365,7.2295,6.84,6.84,-0.6939,-0.6178,-0.50365,-0.3895,-0.77,-5.118265
2,LAUREL JIA EN LIM,27.71,,,6.0,,Discus Throw,,,,...,NaN,NaN,27.71,27.71,NaN,NaN,NaN,NaN,NaN,NaN
3,JOSHUA SHYEN LEE,11.03,,,4.0,,100m,,,,...,10.68120,10.8360,11.03,11.03,-0.6068,-0.5036,-0.34880,-0.1940,-0.71,-1.879845
4,DARYEN XIN TZE KO,54.65,,,2.0,,400m Hurdles,,,,...,NaN,NaN,54.65,54.65,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20241,"Ng, Zavier",9.88m,Hwa Chong Institution,14,18,U15,Triple Jump,0.0,None,Z854G11,...,15.40140,15.1620,9.88,9.88,-5.9204,-5.7608,-5.52140,-5.2820,-6.08,-33.095238
20242,"Choo Jia Yi, Allyson",9.30m,Team Start Singapore,17.0,3,Open,Triple Jump,0.0,None,A967D08,...,12.20725,12.0175,9.3,9.30,-3.2235,-3.0970,-2.90725,-2.7175,-3.35,-21.482213
20243,Lua Yu Xuan,10.02,NYGH,,1.0,C,Triple Jump,,,,...,12.20725,12.0175,10.02,10.02,-2.5035,-2.3770,-2.18725,-1.9975,-2.63,-15.790514
20244,Muhammad Aaryan Shah Bin Azhar,12.61,SSP,,3.0,B,Triple Jump,,,,...,15.40140,15.1620,12.61,12.61,-3.1904,-3.0308,-2.79140,-2.5520,-3.35,-15.989975


In [62]:
# Strip out punctuation from NAME
from itertools import permutations
import string
from dateutil import parser

translator = str.maketrans('', '', string.punctuation)
df['clean_name'] = df['NAME'].str.translate(translator)
df['clean_name'] = df['clean_name'].str.casefold()


df = df.reset_index(drop=True)

translator = str.maketrans('', '', string.punctuation)

df['clean_name'] = df['NAME'].apply(lambda x: str(x).translate(translator))
df['clean_name'] = df['clean_name'].str.casefold()


In [63]:
def parse_dob(x):
    if pd.isna(x) or str(x).strip().lower() in ["none", "nan", "nat", ""]:
        return None
    try:
        # force dayfirst since your DOBs are in DD/MM/YY
        dt = parser.parse(str(x), dayfirst=True)
        # fix 2-digit year misinterpretation (like 1912 instead of 2012)
        if dt.year < 1950:  
            dt = dt.replace(year=dt.year + 100)
        return dt.strftime("%Y-%m-%d")
    except Exception:
        return None

df["DOB_parsed"] = df["DOB"].apply(parse_dob)

# Build dictionary without null DOBs
dictionary_dob_clean_name = {
    row["clean_name"]: row["DOB_parsed"]
    for _, row in df.iterrows()
    if row["DOB_parsed"] is not None
}

# Test Aarika Ray
print(dictionary_dob_clean_name.get("aarika ray"))

2012-05-19


In [64]:

# Read a variation name list and corrections from CSVs
'''
os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/OCTC/')

names = pd.read_csv("name_variations.csv")

for index, row in names.iterrows():
        
    print(names.VARIATION, names.NAME)
    df['NAME'] = df['NAME'].replace(regex=rf"{row['VARIATION']}", value=f"{row['NAME']}")
'''

'\nos.chdir(\'/Users/veesheenyuen/Desktop/DataScience/SAA/OCTC/\')\n\nnames = pd.read_csv("name_variations.csv")\n\nfor index, row in names.iterrows():\n        \n    print(names.VARIATION, names.NAME)\n    df[\'NAME\'] = df[\'NAME\'].replace(regex=rf"{row[\'VARIATION\']}", value=f"{row[\'NAME\']}")\n'

In [65]:
'''
# Read name variations from google sheets

gc = gspread.service_account(filename="/Users/veesheenyuen/Desktop/DataScience/Keys/saa-analytics-7c8937b70609.json")

sheet_name = 'Name_Variations'
sheet_id = '1H3qeiHF1PKzoMG1aIGThvMn3UoBfIHLdLYCXYKOpAbs'

url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/gviz/tq?tqx=out:csv&sheet={sheet_name}"
data = pd.read_csv(url)

data

'''

'\n# Read name variations from google sheets\n\ngc = gspread.service_account(filename="/Users/veesheenyuen/Desktop/DataScience/Keys/saa-analytics-7c8937b70609.json")\n\nsheet_name = \'Name_Variations\'\nsheet_id = \'1H3qeiHF1PKzoMG1aIGThvMn3UoBfIHLdLYCXYKOpAbs\'\n\nurl = f"https://docs.google.com/spreadsheets/d/{sheet_id}/gviz/tq?tqx=out:csv&sheet={sheet_name}"\ndata = pd.read_csv(url)\n\ndata\n\n'

In [66]:
'''

# Read name variations from GCS name lists bucket and replace (Naive, slow version)

# Strip punctuation

#all_punct = string.punctuation # NEW
#excluded_punct = "^$"  # characters to keep NEW

# Create a custom punctuation string without the excluded characters
#custom_punct = ''.join(ch for ch in all_punct if ch not in excluded_punct)

translator = str.maketrans('', '', string.punctuation)
#translator = str.maketrans('', '', custom_punct)  # NEW

#df['NAME'] = df['NAME'].apply(lambda x: str(x).translate(translator)) # NEW



df['NAME'] = df['NAME'].str.replace('\xa0', '', regex=True)
df['NAME'] = df['NAME'].str.replace('[\x00-\x1f\x7f-\x9f]', '', regex=True)
df['NAME'] = df['NAME'].str.replace('\r', '', regex=True)
df['NAME'] = df['NAME'].str.replace('\n', '', regex=True)
df['NAME'] = df['NAME'].str.strip()

df['NAME'] = df['NAME'].str.casefold()  # everything lower case


# Read csv from GCS bucket

file_path = "gs://name_variations/name_variations.csv"
names = pd.read_csv(file_path,
                 sep=",",
                 storage_options={"token": '/Users/veesheenyuen/Desktop/DataScience/Keys/saa-analytics-7c8937b70609.json'})

#names['NAME'] = names['NAME'].apply(lambda x: str(x).translate(translator)) # NEW
#names['VARIATION'] = names['VARIATION'].apply(lambda x: str(x).translate(translator)) #NEW


# Iterate over dataframe and replace names

names['VARIATION'] = names['VARIATION'].str.replace('\xa0', '', regex=True)
names['VARIATION'] = names['VARIATION'].str.replace('[\x00-\x1f\x7f-\x9f]', '', regex=True)
names['VARIATION'] = names['VARIATION'].str.replace('\r', '', regex=True)
names['VARIATION'] = names['VARIATION'].str.replace('\n', '', regex=True)
names['VARIATION'] = names['VARIATION'].str.strip()


names['VARIATION'] = names['VARIATION'].str.casefold() # convert to lower case
names['NAME'] = names['NAME'].str.casefold() # convert to lower case


for row in names.itertuples():  # itertuples is faster
    
    pattern = re.escape(row.VARIATION)  # escapes any special characters in regex before using
  #  df['NAME'] = df['NAME'].replace(regex=rf"{row.VARIATION}", value=f"{row.NAME}")   
    df['NAME'] = df['NAME'].replace(regex=pattern, value=f"{row.NAME}")   

    
df['NAME'] = df['NAME'].str.title()  # capitalize first letter

'''

'\n\n# Read name variations from GCS name lists bucket and replace (Naive, slow version)\n\n# Strip punctuation\n\n#all_punct = string.punctuation # NEW\n#excluded_punct = "^$"  # characters to keep NEW\n\n# Create a custom punctuation string without the excluded characters\n#custom_punct = \'\'.join(ch for ch in all_punct if ch not in excluded_punct)\n\ntranslator = str.maketrans(\'\', \'\', string.punctuation)\n#translator = str.maketrans(\'\', \'\', custom_punct)  # NEW\n\n#df[\'NAME\'] = df[\'NAME\'].apply(lambda x: str(x).translate(translator)) # NEW\n\n\n\ndf[\'NAME\'] = df[\'NAME\'].str.replace(\'\xa0\', \'\', regex=True)\ndf[\'NAME\'] = df[\'NAME\'].str.replace(\'[\x00-\x1f\x7f-\x9f]\', \'\', regex=True)\ndf[\'NAME\'] = df[\'NAME\'].str.replace(\'\r\', \'\', regex=True)\ndf[\'NAME\'] = df[\'NAME\'].str.replace(\'\n\', \'\', regex=True)\ndf[\'NAME\'] = df[\'NAME\'].str.strip()\n\ndf[\'NAME\'] = df[\'NAME\'].str.casefold()  # everything lower case\n\n\n# Read csv from GCS bucket\

In [67]:
'''

# 2nd variation replacement code (moderate speed and robustness)

def normalize_text(s):
    return (str(s)
            .replace('\xa0', '')
            .replace('\r', '')
            .replace('\n', '')
            .strip()
            .casefold())

# Normalize your dataframe names
df['NAME'] = df['NAME'].apply(normalize_text)


# Load and normalize variations file

file_path = "gs://name_variations/name_variations.csv"

names = pd.read_csv(file_path,
                    sep=',',
                    storage_options={"token": '/Users/veesheenyuen/Desktop/DataScience/Keys/saa-analytics-7c8937b70609.json'})

names['VARIATION'] = names['VARIATION'].apply(normalize_text)
names['NAME'] = names['NAME'].apply(normalize_text)

# Apply regex replacements without escaping VARIATION column
#for row in names.itertuples():
#    df['NAME'] = df['NAME'].str.replace(row.VARIATION, row.NAME, regex=True)

for row in names.itertuples():
    pattern = row.VARIATION
    try:
        re.compile(pattern)  # test compilation first
        df['NAME'] = df['NAME'].str.replace(pattern, row.NAME, regex=True)
    except re.error as e:
        print(f"Invalid regex pattern: {pattern} | Error: {e}")

# Capitalize final standardized names
df['NAME'] = df['NAME'].str.title()

'''

'\n\n# 2nd variation replacement code (moderate speed and robustness)\n\ndef normalize_text(s):\n    return (str(s)\n            .replace(\'\xa0\', \'\')\n            .replace(\'\r\', \'\')\n            .replace(\'\n\', \'\')\n            .strip()\n            .casefold())\n\n# Normalize your dataframe names\ndf[\'NAME\'] = df[\'NAME\'].apply(normalize_text)\n\n\n# Load and normalize variations file\n\nfile_path = "gs://name_variations/name_variations.csv"\n\nnames = pd.read_csv(file_path,\n                    sep=\',\',\n                    storage_options={"token": \'/Users/veesheenyuen/Desktop/DataScience/Keys/saa-analytics-7c8937b70609.json\'})\n\nnames[\'VARIATION\'] = names[\'VARIATION\'].apply(normalize_text)\nnames[\'NAME\'] = names[\'NAME\'].apply(normalize_text)\n\n# Apply regex replacements without escaping VARIATION column\n#for row in names.itertuples():\n#    df[\'NAME\'] = df[\'NAME\'].str.replace(row.VARIATION, row.NAME, regex=True)\n\nfor row in names.itertuples():\n  

In [68]:
# Fastest execution speed version

import pandas as pd
import re

# Normalize function as before
def normalize_text(s):
    return (str(s)
            .replace('\xa0', '')
            .replace('\r', '')
            .replace('\n', '')
            .strip()
            .casefold())

# Normalize dataframe
df['NAME'] = df['NAME'].apply(normalize_text)

# Load variations file and normalize
file_path = "gs://name_variations/name_variations.csv"
names = pd.read_csv(file_path,
                    sep=',',
                    storage_options={"token": '/Users/veesheenyuen/Desktop/DataScience/Keys/saa-analytics-7c8937b70609.json'})

names['VARIATION'] = names['VARIATION'].apply(normalize_text)
names['NAME'] = names['NAME'].apply(normalize_text)

# Precompile all regex patterns safely

compiled_patterns = []
for pattern_str, replacement in zip(names['VARIATION'], names['NAME']):
    try:
        compiled_re = re.compile(pattern_str)
        compiled_patterns.append( (compiled_re, replacement) )
    except re.error as e:
        print(f"Skipping invalid regex pattern: {pattern_str} Error: {e}")

# Iterate over all patterns and apply replacements using precompiled regexes
for regex, replacement in compiled_patterns:
    df['NAME'] = df['NAME'].str.replace(regex, replacement, regex=True)

# Capitalize final standardized names
df['NAME'] = df['NAME'].str.title()


In [69]:
compiled_patterns

[(re.compile(r'soh, aidan michael zi ren', re.UNICODE),
  'aidan michael soh zi ren'),
 (re.compile(r'aidan michael soh zi ren', re.UNICODE),
  'aidan michael soh zi ren'),
 (re.compile(r'soh zi ren, aidan michael', re.UNICODE),
  'aidan michael soh zi ren'),
 (re.compile(r'soh zi ren, aidan michael', re.UNICODE),
  'aidan michael soh zi ren'),
 (re.compile(r'soh, aidan michael', re.UNICODE), 'aidan michael soh zi ren'),
 (re.compile(r'., aidan michael soh zi', re.UNICODE),
  'aidan michael soh zi ren'),
 (re.compile(r'^aidan michael soh$', re.UNICODE), 'aidan michael soh zi ren'),
 (re.compile(r'ang, chen xiang', re.UNICODE), 'ang chen xiang'),
 (re.compile(r'cheng xiang ang', re.UNICODE), 'ang chen xiang'),
 (re.compile(r'chen, xiang', re.UNICODE), 'ang chen xiang'),
 (re.compile(r'chen xiang', re.UNICODE), 'ang chen xiang'),
 (re.compile(r'^ang, chen xiang$', re.UNICODE), 'ang chen xiang'),
 (re.compile(r'^chen xiang ang$', re.UNICODE), 'ang chen xiang'),
 (re.compile(r'., ang chen 

In [70]:
df

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT_x,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,RESULT_CONV,RESULT_BEST,Delta1,Delta2,Delta3.5,Delta5,Delta_Benchmark,PERF_SCALAR,clean_name,DOB_parsed
0,Si En Tabitha Ng,11:03.95,,,4.0,,3000m,,,,...,663.95,663.95,-62.9091,-56.9582,-48.03185,-39.1055,-68.86,-6.571359,si en tabitha ng,None
1,Je An Chua Garrett,6.84,,,3.0,,Long Jump,,,,...,6.84,6.84,-0.6939,-0.6178,-0.50365,-0.3895,-0.77,-5.118265,je an garrett chua,None
2,Laurel Jia En Lim,27.71,,,6.0,,Discus Throw,,,,...,27.71,27.71,NaN,NaN,NaN,NaN,NaN,NaN,laurel jia en lim,None
3,Lee Joshua Shyen,11.03,,,4.0,,100m,,,,...,11.03,11.03,-0.6068,-0.5036,-0.34880,-0.1940,-0.71,-1.879845,joshua shyen lee,None
4,Daryen Xin Tze Ko,54.65,,,2.0,,400m Hurdles,,,,...,54.65,54.65,NaN,NaN,NaN,NaN,NaN,NaN,daryen xin tze ko,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20241,"Ng, Zavier",9.88m,Hwa Chong Institution,14,18,U15,Triple Jump,0.0,None,Z854G11,...,9.88,9.88,-5.9204,-5.7608,-5.52140,-5.2820,-6.08,-33.095238,ng zavier,2011-03-14
20242,"Choo Jia Yi, Allyson",9.30m,Team Start Singapore,17.0,3,Open,Triple Jump,0.0,None,A967D08,...,9.3,9.30,-3.2235,-3.0970,-2.90725,-2.7175,-3.35,-21.482213,choo jia yi allyson,2008-01-10
20243,Lua Yu Xuan,10.02,NYGH,,1.0,C,Triple Jump,,,,...,10.02,10.02,-2.5035,-2.3770,-2.18725,-1.9975,-2.63,-15.790514,lua yu xuan,None
20244,Muhammad Aaryan Shah Bin Azhar,12.61,SSP,,3.0,B,Triple Jump,,,,...,12.61,12.61,-3.1904,-3.0308,-2.79140,-2.5520,-3.35,-15.989975,muhammad aaryan shah bin azhar,None


In [71]:
# Exclude foreigners from MALAYSIA, THAILAND etc.

#df_select = df[(df['TEAM']!='Malaysia') & (df['TEAM']!='THAILAND') & (df['TEAM']!='China') & (df['TEAM']!='South Korea') & (df['TEAM']!='Laos') & (df['TEAM']!='Philippines') & (df['TEAM']!='Piboonbumpen Thailand') & (df['TEAM']!='Chinese Taipei') & (df['TEAM']!='Gurkha Contingent') & (df['TEAM']!='Australia') & (df['TEAM']!='Piboonbumpen Thailand') & (df['TEAM']!='Hong Kong') & (df['TEAM']!='PERAK')] 

df_select = df[(df['TEAM']!='Malaysia')&(df['TEAM']!='THAILAND')&(df['TEAM']!='China')&(df['TEAM']!='Thailand') 
                       &(df['TEAM']!='South Korea')&(df['TEAM']!='Laos')&(df['TEAM']!='Myanmar') 
                       &(df['TEAM']!='Philippines')&(df['TEAM']!='Piboonbumpen Thailand') 
                       &(df['TEAM']!='Chinese Taipei')&(df['TEAM']!='Gurkha Contingent') 
                       &(df['TEAM']!='Australia')&(df['TEAM']!='Piboonbumpen Thailand') 
                       &(df['TEAM']!='Hong Kong')&(df['TEAM']!='PERAK')&(df['TEAM']!='Sri Lanka') 
                       &(df['TEAM']!='Indonesia')&(df['TEAM']!='THAILAND')&(df['TEAM']!='MALAYSIA') 
                       &(df['TEAM']!='PHILIPPINES') & (df['TEAM']!='SOUTH KOREA')&(df['TEAM']!='Waseda') 
                       &(df['TEAM']!='LAOS')&(df['TEAM']!='CHINESE TAIPEI')&(df['TEAM']!='Vietnam')
                       &(df['TEAM']!='INDIA')&(df['TEAM']!='Hong Kong, China')&(df['TEAM']!='AIC JAPAN')
                       &(df['NATIONALITY']!='GBR')&(df['NATIONALITY']!='JPN')&(df['NATIONALITY']!='SRI')&(df['NATIONALITY']!='SAM')
                       &(df['NATIONALITY']!='THA')&(df['NATIONALITY']!='IND')] 

In [72]:
df_select

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT_x,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,RESULT_CONV,RESULT_BEST,Delta1,Delta2,Delta3.5,Delta5,Delta_Benchmark,PERF_SCALAR,clean_name,DOB_parsed
0,Si En Tabitha Ng,11:03.95,,,4.0,,3000m,,,,...,663.95,663.95,-62.9091,-56.9582,-48.03185,-39.1055,-68.86,-6.571359,si en tabitha ng,None
1,Je An Chua Garrett,6.84,,,3.0,,Long Jump,,,,...,6.84,6.84,-0.6939,-0.6178,-0.50365,-0.3895,-0.77,-5.118265,je an garrett chua,None
2,Laurel Jia En Lim,27.71,,,6.0,,Discus Throw,,,,...,27.71,27.71,NaN,NaN,NaN,NaN,NaN,NaN,laurel jia en lim,None
3,Lee Joshua Shyen,11.03,,,4.0,,100m,,,,...,11.03,11.03,-0.6068,-0.5036,-0.34880,-0.1940,-0.71,-1.879845,joshua shyen lee,None
4,Daryen Xin Tze Ko,54.65,,,2.0,,400m Hurdles,,,,...,54.65,54.65,NaN,NaN,NaN,NaN,NaN,NaN,daryen xin tze ko,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20241,"Ng, Zavier",9.88m,Hwa Chong Institution,14,18,U15,Triple Jump,0.0,None,Z854G11,...,9.88,9.88,-5.9204,-5.7608,-5.52140,-5.2820,-6.08,-33.095238,ng zavier,2011-03-14
20242,"Choo Jia Yi, Allyson",9.30m,Team Start Singapore,17.0,3,Open,Triple Jump,0.0,None,A967D08,...,9.3,9.30,-3.2235,-3.0970,-2.90725,-2.7175,-3.35,-21.482213,choo jia yi allyson,2008-01-10
20243,Lua Yu Xuan,10.02,NYGH,,1.0,C,Triple Jump,,,,...,10.02,10.02,-2.5035,-2.3770,-2.18725,-1.9975,-2.63,-15.790514,lua yu xuan,None
20244,Muhammad Aaryan Shah Bin Azhar,12.61,SSP,,3.0,B,Triple Jump,,,,...,12.61,12.61,-3.1904,-3.0308,-2.79140,-2.5520,-3.35,-15.989975,muhammad aaryan shah bin azhar,None


In [73]:
'''
# Read list of foreigners

foreigners = pd.read_csv('/Users/veesheenyuen/Desktop/DataScience/SAA/MM/List of Foreigners.csv', encoding='latin-1')
'''


"\n# Read list of foreigners\n\nforeigners = pd.read_csv('/Users/veesheenyuen/Desktop/DataScience/SAA/MM/List of Foreigners.csv', encoding='latin-1')\n"

In [74]:
# Read list of foreigners from GCS bucket

file_path = "gs://name_lists/List of Foreigners.csv"
foreigners = pd.read_csv(file_path,
                 sep=",",
                 encoding="unicode escape",
                 storage_options={"token": '/Users/veesheenyuen/Desktop/DataScience/Keys/saa-analytics-7c8937b70609.json'})


In [75]:
foreigners

,LAST_NAME,FIRST_NAME
0,Aaryan,Greuter Christoph
1,Akahodani,Takayuki
2,Apondar,Audric
3,Brooks,Ruby
4,Brouwer,Cees
...,...,...
235,Kashama,Biwesa Daniel
236,ISMAIL,MUHAMMAD ZULFIQAR
237,Jayaganeson,Kirtisha
238,LIN,Yu Sian


In [76]:
df_select['NAME'].str.casefold()

0                      si en tabitha ng
1                    je an chua garrett
2                     laurel jia en lim
3                      lee joshua shyen
4                     daryen xin tze ko
                      ...              
20241                        ng, zavier
20242              choo jia yi, allyson
20243                       lua yu xuan
20244    muhammad aaryan shah bin azhar
20245                   carrie-anne loh
Name: NAME, Length: 19975, dtype: object

In [77]:
foreigners['V1'] = foreigners['LAST_NAME']+' '+foreigners['FIRST_NAME']
foreigners['V2'] = foreigners['FIRST_NAME']+' '+foreigners['LAST_NAME']
foreigners['V3'] = foreigners['LAST_NAME']+', '+foreigners['FIRST_NAME']
foreigners['V4'] = foreigners['FIRST_NAME']+' '+foreigners['LAST_NAME']

for1 = foreigners['V1'].dropna().tolist()
for2 = foreigners['V2'].dropna().tolist()
for3 = foreigners['V3'].dropna().tolist()
for4 = foreigners['V4'].dropna().tolist()

foreign_list = for1+for2+for3+for4 

foreign_list_casefold=[s.casefold() for s in foreign_list]

exclusions = foreign_list_casefold

no_foreigners_list = df_select.loc[~df['NAME'].str.casefold().isin(exclusions)]  # ~ means NOT IN. DROP spex carded athletes

In [78]:
no_foreigners_list

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT_x,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,RESULT_CONV,RESULT_BEST,Delta1,Delta2,Delta3.5,Delta5,Delta_Benchmark,PERF_SCALAR,clean_name,DOB_parsed
0,Si En Tabitha Ng,11:03.95,,,4.0,,3000m,,,,...,663.95,663.95,-62.9091,-56.9582,-48.03185,-39.1055,-68.86,-6.571359,si en tabitha ng,None
1,Je An Chua Garrett,6.84,,,3.0,,Long Jump,,,,...,6.84,6.84,-0.6939,-0.6178,-0.50365,-0.3895,-0.77,-5.118265,je an garrett chua,None
2,Laurel Jia En Lim,27.71,,,6.0,,Discus Throw,,,,...,27.71,27.71,NaN,NaN,NaN,NaN,NaN,NaN,laurel jia en lim,None
3,Lee Joshua Shyen,11.03,,,4.0,,100m,,,,...,11.03,11.03,-0.6068,-0.5036,-0.34880,-0.1940,-0.71,-1.879845,joshua shyen lee,None
4,Daryen Xin Tze Ko,54.65,,,2.0,,400m Hurdles,,,,...,54.65,54.65,NaN,NaN,NaN,NaN,NaN,NaN,daryen xin tze ko,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20241,"Ng, Zavier",9.88m,Hwa Chong Institution,14,18,U15,Triple Jump,0.0,None,Z854G11,...,9.88,9.88,-5.9204,-5.7608,-5.52140,-5.2820,-6.08,-33.095238,ng zavier,2011-03-14
20242,"Choo Jia Yi, Allyson",9.30m,Team Start Singapore,17.0,3,Open,Triple Jump,0.0,None,A967D08,...,9.3,9.30,-3.2235,-3.0970,-2.90725,-2.7175,-3.35,-21.482213,choo jia yi allyson,2008-01-10
20243,Lua Yu Xuan,10.02,NYGH,,1.0,C,Triple Jump,,,,...,10.02,10.02,-2.5035,-2.3770,-2.18725,-1.9975,-2.63,-15.790514,lua yu xuan,None
20244,Muhammad Aaryan Shah Bin Azhar,12.61,SSP,,3.0,B,Triple Jump,,,,...,12.61,12.61,-3.1904,-3.0308,-2.79140,-2.5520,-3.35,-15.989975,muhammad aaryan shah bin azhar,None


In [79]:
'''
# Choose the best result for each event participated by every athlete (original version)

top_performers_clean = no_foreigners_list.sort_values(['MAPPED_EVENT', 'NAME','PERF_SCALAR'],ascending=False).groupby(['MAPPED_EVENT', 'NAME']).head(1)
'''

"\n# Choose the best result for each event participated by every athlete (original version)\n\ntop_performers_clean = no_foreigners_list.sort_values(['MAPPED_EVENT', 'NAME','PERF_SCALAR'],ascending=False).groupby(['MAPPED_EVENT', 'NAME']).head(1)\n"

In [80]:
# Choose best performance per event and athlete (robust version)

# 1) Convert to numeric FIRST
no_foreigners_list["PERF_SCALAR"] = pd.to_numeric(no_foreigners_list["PERF_SCALAR"], errors="coerce")

# 2) Drop non-numeric (NaNs) and non-finite values
df2 = no_foreigners_list[np.isfinite(no_foreigners_list["PERF_SCALAR"])].copy()

# 3) Normalize grouping keys to avoid silent split by stray spaces
for col in ["NAME", "MAPPED_EVENT"]:
    df2[col] = df2[col].astype(str).str.strip()

# 4) Keep the best (largest) per group
top_performers_clean = (
    df2.sort_values(
        ["MAPPED_EVENT", "NAME", "PERF_SCALAR"],
        ascending=[True, True, False]
    )
    .drop_duplicates(subset=["MAPPED_EVENT", "NAME"], keep="first")
    .reset_index(drop=True)
)

top_performers_clean

/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_49156/3454415374.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  no_foreigners_list["PERF_SCALAR"] = pd.to_numeric(no_foreigners_list["PERF_SCALAR"], errors="coerce")


,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT_x,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,RESULT_CONV,RESULT_BEST,Delta1,Delta2,Delta3.5,Delta5,Delta_Benchmark,PERF_SCALAR,clean_name,DOB_parsed
0,'Aqielah Binte Zahrin,00:17.02,JY,,86.0,C,100m,,,,...,17.02,17.02,-5.2636,-5.1472,-4.97260,-4.7980,-5.38,-41.219931,aqielah binte zahrin,None
1,"., Elfirman",0:13.78,NJAC Malaysia,12.0,7,U13,Dash,100.0,(22 Nov),None,...,13.78,13.78,-3.3568,-3.2536,-3.09880,-2.9440,-3.46,-28.527132,elfirman,2013-02-05
2,"., Sarah",0:15.29,NJAC Malaysia,12.0,10,U13,Dash,100.0,(22 Nov),None,...,15.29,15.29,-3.5336,-3.4172,-3.24260,-3.0680,-3.65,-26.357388,sarah,2013-08-23
3,A A Ayu Teja Gayatri,00:17.02,SJC,,87.0,B,100m,,,,...,17.02,17.02,-5.2636,-5.1472,-4.97260,-4.7980,-5.38,-41.219931,a a ayu teja gayatri,None
4,Aadil Zulhaqeem Zulkifli,16.58,Erovra Club -,,33,,100m,,nan,,...,16.58,16.58,-6.1568,-6.0536,-5.89880,-5.7440,-6.26,-55.658915,aadil zulhaqeem zulkifli,2012-03-22
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7769,Zhao Daniel,11.98,-,,1,,Triple Jump,,nan,,...,11.98,11.98,-3.8204,-3.6608,-3.42140,-3.1820,-3.98,-19.937343,daniel zhao,2011-07-25
7770,"Zhao, Daniel",11.67m,Hwa Chong Institution,14,3,U15,Triple Jump,0.0,None,None,...,11.67,11.67,-4.1304,-3.9708,-3.73140,-3.4920,-4.29,-21.879699,zhao daniel,2011-07-25
7771,Zheng Justin De,10.47m,National Junior College,14,12,U15,Triple Jump,0.0,None,J034B11,...,10.47,10.47,-5.3304,-5.1708,-4.93140,-4.6920,-5.49,-29.398496,zheng justin de,2011-02-21
7772,Zhong Chuhan,12.26,,,,,Triple Jump,,,,...,12.26,12.26,-0.2635,-0.1370,0.05275,0.2425,-0.39,1.916996,chuhan zhong,2005-10-18


In [81]:
top_performers_clean.reset_index(inplace=True)


In [82]:
top_performers_clean

,index,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT_x,DISTANCE,EVENT_CLASS,...,RESULT_CONV,RESULT_BEST,Delta1,Delta2,Delta3.5,Delta5,Delta_Benchmark,PERF_SCALAR,clean_name,DOB_parsed
0,0,'Aqielah Binte Zahrin,00:17.02,JY,,86.0,C,100m,,,...,17.02,17.02,-5.2636,-5.1472,-4.97260,-4.7980,-5.38,-41.219931,aqielah binte zahrin,None
1,1,"., Elfirman",0:13.78,NJAC Malaysia,12.0,7,U13,Dash,100.0,(22 Nov),...,13.78,13.78,-3.3568,-3.2536,-3.09880,-2.9440,-3.46,-28.527132,elfirman,2013-02-05
2,2,"., Sarah",0:15.29,NJAC Malaysia,12.0,10,U13,Dash,100.0,(22 Nov),...,15.29,15.29,-3.5336,-3.4172,-3.24260,-3.0680,-3.65,-26.357388,sarah,2013-08-23
3,3,A A Ayu Teja Gayatri,00:17.02,SJC,,87.0,B,100m,,,...,17.02,17.02,-5.2636,-5.1472,-4.97260,-4.7980,-5.38,-41.219931,a a ayu teja gayatri,None
4,4,Aadil Zulhaqeem Zulkifli,16.58,Erovra Club -,,33,,100m,,nan,...,16.58,16.58,-6.1568,-6.0536,-5.89880,-5.7440,-6.26,-55.658915,aadil zulhaqeem zulkifli,2012-03-22
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7769,7769,Zhao Daniel,11.98,-,,1,,Triple Jump,,nan,...,11.98,11.98,-3.8204,-3.6608,-3.42140,-3.1820,-3.98,-19.937343,daniel zhao,2011-07-25
7770,7770,"Zhao, Daniel",11.67m,Hwa Chong Institution,14,3,U15,Triple Jump,0.0,None,...,11.67,11.67,-4.1304,-3.9708,-3.73140,-3.4920,-4.29,-21.879699,zhao daniel,2011-07-25
7771,7771,Zheng Justin De,10.47m,National Junior College,14,12,U15,Triple Jump,0.0,None,...,10.47,10.47,-5.3304,-5.1708,-4.93140,-4.6920,-5.49,-29.398496,zheng justin de,2011-02-21
7772,7772,Zhong Chuhan,12.26,,,,,Triple Jump,,,...,12.26,12.26,-0.2635,-0.1370,0.05275,0.2425,-0.39,1.916996,chuhan zhong,2005-10-18


In [83]:
os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/Asian Indoor/')


top_performers_clean.to_csv('asian_indoor_top_performers_jan3.csv', encoding='utf-8')

In [84]:
# Choose best performance for each event

#tiered_performers = top_performers_clean.sort_values(['GENDER', 'MAPPED_EVENT', 'PERF_SCALAR'],ascending=False).groupby(['MAPPED_EVENT', 'NAME']).head(1)

tiered_performers = top_performers_clean


In [85]:
tiered_performers

,index,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT_x,DISTANCE,EVENT_CLASS,...,RESULT_CONV,RESULT_BEST,Delta1,Delta2,Delta3.5,Delta5,Delta_Benchmark,PERF_SCALAR,clean_name,DOB_parsed
0,0,'Aqielah Binte Zahrin,00:17.02,JY,,86.0,C,100m,,,...,17.02,17.02,-5.2636,-5.1472,-4.97260,-4.7980,-5.38,-41.219931,aqielah binte zahrin,None
1,1,"., Elfirman",0:13.78,NJAC Malaysia,12.0,7,U13,Dash,100.0,(22 Nov),...,13.78,13.78,-3.3568,-3.2536,-3.09880,-2.9440,-3.46,-28.527132,elfirman,2013-02-05
2,2,"., Sarah",0:15.29,NJAC Malaysia,12.0,10,U13,Dash,100.0,(22 Nov),...,15.29,15.29,-3.5336,-3.4172,-3.24260,-3.0680,-3.65,-26.357388,sarah,2013-08-23
3,3,A A Ayu Teja Gayatri,00:17.02,SJC,,87.0,B,100m,,,...,17.02,17.02,-5.2636,-5.1472,-4.97260,-4.7980,-5.38,-41.219931,a a ayu teja gayatri,None
4,4,Aadil Zulhaqeem Zulkifli,16.58,Erovra Club -,,33,,100m,,nan,...,16.58,16.58,-6.1568,-6.0536,-5.89880,-5.7440,-6.26,-55.658915,aadil zulhaqeem zulkifli,2012-03-22
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7769,7769,Zhao Daniel,11.98,-,,1,,Triple Jump,,nan,...,11.98,11.98,-3.8204,-3.6608,-3.42140,-3.1820,-3.98,-19.937343,daniel zhao,2011-07-25
7770,7770,"Zhao, Daniel",11.67m,Hwa Chong Institution,14,3,U15,Triple Jump,0.0,None,...,11.67,11.67,-4.1304,-3.9708,-3.73140,-3.4920,-4.29,-21.879699,zhao daniel,2011-07-25
7771,7771,Zheng Justin De,10.47m,National Junior College,14,12,U15,Triple Jump,0.0,None,...,10.47,10.47,-5.3304,-5.1708,-4.93140,-4.6920,-5.49,-29.398496,zheng justin de,2011-02-21
7772,7772,Zhong Chuhan,12.26,,,,,Triple Jump,,,...,12.26,12.26,-0.2635,-0.1370,0.05275,0.2425,-0.39,1.916996,chuhan zhong,2005-10-18


In [86]:
# Identify Tier 1/2/3 performers

#top_performers_clean['TIER'] = np.where((top_performers_clean['Delta_Benchmark']>=0), 'Tier 1',    
#                                np.where(((top_performers_clean['Delta_Benchmark']<0) & (top_performers_clean['Delta2']>=0)), 'Tier2',
#                                np.where(((top_performers_clean['Delta2']<0) & (top_performers_clean['Delta3.5']>=0)), 'Tier3', ' ')))


tiered_performers['TIER'] = np.where((tiered_performers['Delta_Benchmark']>=0), 'Tier 1',    
                                np.where(((tiered_performers['Delta_Benchmark']<0) & (tiered_performers['Delta1']>=0)), 'Tier 2',
                                np.where(((tiered_performers['Delta1']<0) & (tiered_performers['Delta2']>=0)), 'Tier 3',
                                np.where(((tiered_performers['Delta2']<0) & (tiered_performers['Delta3.5']>=0)), 'Tier 4',
                                np.where(((tiered_performers['Delta3.5']<0) & (tiered_performers['Delta5']>=0)), 'Tier 5', ' ')))))



In [87]:
# Drop rows without a SEAG benchmark

final_df = tiered_performers[tiered_performers['BENCHMARK'].notna()]


In [88]:
final_df

,index,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT_x,DISTANCE,EVENT_CLASS,...,RESULT_BEST,Delta1,Delta2,Delta3.5,Delta5,Delta_Benchmark,PERF_SCALAR,clean_name,DOB_parsed,TIER
0,0,'Aqielah Binte Zahrin,00:17.02,JY,,86.0,C,100m,,,...,17.02,-5.2636,-5.1472,-4.97260,-4.7980,-5.38,-41.219931,aqielah binte zahrin,None,
1,1,"., Elfirman",0:13.78,NJAC Malaysia,12.0,7,U13,Dash,100.0,(22 Nov),...,13.78,-3.3568,-3.2536,-3.09880,-2.9440,-3.46,-28.527132,elfirman,2013-02-05,
2,2,"., Sarah",0:15.29,NJAC Malaysia,12.0,10,U13,Dash,100.0,(22 Nov),...,15.29,-3.5336,-3.4172,-3.24260,-3.0680,-3.65,-26.357388,sarah,2013-08-23,
3,3,A A Ayu Teja Gayatri,00:17.02,SJC,,87.0,B,100m,,,...,17.02,-5.2636,-5.1472,-4.97260,-4.7980,-5.38,-41.219931,a a ayu teja gayatri,None,
4,4,Aadil Zulhaqeem Zulkifli,16.58,Erovra Club -,,33,,100m,,nan,...,16.58,-6.1568,-6.0536,-5.89880,-5.7440,-6.26,-55.658915,aadil zulhaqeem zulkifli,2012-03-22,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7769,7769,Zhao Daniel,11.98,-,,1,,Triple Jump,,nan,...,11.98,-3.8204,-3.6608,-3.42140,-3.1820,-3.98,-19.937343,daniel zhao,2011-07-25,
7770,7770,"Zhao, Daniel",11.67m,Hwa Chong Institution,14,3,U15,Triple Jump,0.0,None,...,11.67,-4.1304,-3.9708,-3.73140,-3.4920,-4.29,-21.879699,zhao daniel,2011-07-25,
7771,7771,Zheng Justin De,10.47m,National Junior College,14,12,U15,Triple Jump,0.0,None,...,10.47,-5.3304,-5.1708,-4.93140,-4.6920,-5.49,-29.398496,zheng justin de,2011-02-21,
7772,7772,Zhong Chuhan,12.26,,,,,Triple Jump,,,...,12.26,-0.2635,-0.1370,0.05275,0.2425,-0.39,1.916996,chuhan zhong,2005-10-18,Tier 4


In [89]:
# Process dates to extract age

# Map NSG divisions into age

mask = (final_df['DIVISION'].str.contains(r'A', na=False))
final_df.loc[mask, 'AGE'] = '18.5'

mask = (final_df['DIVISION'].str.contains(r'B', na=False))
final_df.loc[mask, 'AGE'] = '16'

mask = (final_df['DIVISION'].str.contains(r'C', na=False))
final_df.loc[mask, 'AGE'] = '13.5'

mask = (final_df['DIVISION'].str.contains(r'O', na=False))
final_df.loc[mask, 'AGE'] = '12'



In [90]:
def length(string):

    B = ''
    year = ''

    try:

        length = len(string)

        if length == 2:

            string = '19' + string

        elif length == 1:

            string = ''

        else:

            pass

        if string is not None or len(string) != 1:

            B = parser.parse(string, dayfirst=True)
                        
    except:

        pass

    return B


final_df['DOB_new'] = final_df['DOB'].apply(length)



#B = parser.parse("10-09-2021", dayfirst = True)

In [91]:
final_df['DOB_new'] = pd.to_datetime(final_df['DOB_new'], errors='coerce')

final_df['year_extract']=final_df['DOB_new'].dt.strftime('%Y')

final_df['year_extract'] = pd.to_numeric(final_df['year_extract'])

final_df['age_extract'] = 2025 - final_df['year_extract']


In [92]:
def age(number):  # correct negative age numbers
    
    if number<0:
        
        number+=100
        
    return number


final_df['age_extract']=final_df['age_extract'].apply(age)


In [93]:
# If NSG event then choose AGE otherwise choose age_extract

condition1 = final_df['COMPETITION_x']=='National School Games'
#condition2=((df['CATEGORY_EVENT']=='Jump')|(df['CATEGORY_EVENT']=='Throw'))
#condition3=df['SEED_CONV']<df['RESULT_CONV']
#condition4=~((df['CATEGORY_EVENT']=='Jump')|(df['CATEGORY_EVENT']=='Throw'))


final_df['age_extract'] = final_df['AGE'].where((condition1), final_df['age_extract'].values)


In [94]:
# Change to numeric

final_df[['age_extract']] = final_df[['age_extract']].apply(pd.to_numeric)

In [95]:
# Remove relay events

final_df = final_df[final_df['MAPPED_EVENT']!='4 x 100m']

In [96]:
final_tiered_selection = final_df[final_df['TIER']!=' ']

In [97]:
final_tiered_selection.columns

Index(['index', 'NAME', 'RESULT', 'TEAM', 'AGE', 'COMPETITION_RANK',
       'DIVISION', 'EVENT_x', 'DISTANCE', 'EVENT_CLASS', 'UNIQUE_ID', 'DOB',
       'NATIONALITY', 'WIND', 'CATEGORY_EVENT', 'GENDER', 'COMPETITION_x',
       'DATE', 'YEAR', 'REGION', 'TIMESTAMP', 'NOW', 'delta_time',
       'delta_time_conv', 'event_month', 'MAPPED_EVENT', 'COMPETITION_y',
       'EVENT_y', 'BENCHMARK', 'BENCHMARK_C', '1%', '2%', '3.5%', '5%',
       'RESULT_CONV', 'RESULT_BEST', 'Delta1', 'Delta2', 'Delta3.5', 'Delta5',
       'Delta_Benchmark', 'PERF_SCALAR', 'clean_name', 'DOB_parsed', 'TIER',
       'DOB_new', 'year_extract', 'age_extract'],
      dtype='object')

In [98]:
# Convert time format for marathon and 5000m into mm:ss.00
# Choose the correct column indices or you will get erratic timings

import datetime

#s=247.779

#datetime.datetime.fromtimestamp(s).strftime('%M:%S.%f')

all_ranking=final_tiered_selection.reset_index(drop=True)


#all_ranking[['2%', '3.5%', '5%']] = df[['2%', '3.5%', '5%']].apply(pd.to_numeric)


#all_ranking['2%'] = all_ranking['2%'].astype("string")
#all_ranking['3.5%'] = all_ranking['3.5%'].astype("string")
#all_ranking['5%'] = all_ranking['5%'].astype("string")


for i in range(len(all_ranking)):
        
    rowIndex = all_ranking.index[i]

    event=all_ranking.loc[rowIndex,'MAPPED_EVENT']
        
    
#    time_base2=all_ranking.iloc[rowIndex,25]
#    time_base3=all_ranking.iloc[rowIndex,26]
#    time_base5=all_ranking.iloc[rowIndex,27]
#    time_base10=all_ranking.iloc[rowIndex,27]
    time_base2=all_ranking.loc[rowIndex,'2%']
    time_base3=all_ranking.loc[rowIndex,'3.5%']
    time_base5=all_ranking.loc[rowIndex,'5%']
 #   time_base10=all_ranking.loc[rowIndex,'10%']
    
        
    if all_ranking.loc[rowIndex,'BENCHMARK_C']==None:
        continue
        
    if event=='800m' or event=='10,000m' or event=='5000m' or event=='3000m Steeplechase' or event=='1500m':
        
      #  print(i, event, time_base2, time_base3, time_base5)
   
        
        date_preconvert2 = datetime.datetime.utcfromtimestamp(time_base2)
        date_preconvert3 = datetime.datetime.utcfromtimestamp(time_base3)
        date_preconvert5 = datetime.datetime.utcfromtimestamp(time_base5)
   #     date_preconvert10 = datetime.datetime.utcfromtimestamp(time_base10)

        
    #    print(date_preconvert2, date_preconvert3, date_preconvert5)
            
        
        output2 = datetime.datetime.strftime(date_preconvert2, "%M:%S.%f")
        output3 = datetime.datetime.strftime(date_preconvert3, "%M:%S.%f")
        output5 = datetime.datetime.strftime(date_preconvert5, "%M:%S.%f")
 #       output10 = datetime.datetime.strftime(date_preconvert10, "%M:%S.%f")

        
     #   print(event, output2, output3, output5)

                    
       #     top_performers_clean.loc[rowIndex, '2%_timing'] = output2
       #     top_performers_clean.loc[rowIndex, '3.5%_timing'] = output3
       #     top_performers_clean.loc[rowIndex, '5%_timing'] = output5
            
   
        all_ranking.at[rowIndex, '2%'] = output2 # copy over time format
        all_ranking.at[rowIndex, '3.5%'] = output3
        all_ranking.at[rowIndex, '5%'] = output5
  #      all_ranking.at[rowIndex, '10%'] = output10

        
    elif event=='Marathon':
        
      #  print(time_base2, time_base3, time_base5)

        
        try:
            

        
            date_preconvert2 = datetime.datetime.utcfromtimestamp(time_base2)
            date_preconvert3 = datetime.datetime.utcfromtimestamp(time_base3)
            date_preconvert5 = datetime.datetime.utcfromtimestamp(time_base5)
     #       date_preconvert10 = datetime.datetime.utcfromtimestamp(time_base10)

            
            
            output2 = datetime.datetime.strftime(date_preconvert2, "%H:%M:%S")
            output3 = datetime.datetime.strftime(date_preconvert3, "%H:%M:%S")
            output5 = datetime.datetime.strftime(date_preconvert5, "%H:%M:%S")
    #        output10 = datetime.datetime.strftime(date_preconvert10, "%H:%M:%S")

            
        
        #    top_performers_clean.loc[rowIndex, '2%_timing'] = output2
        #    top_performers_clean.loc[rowIndex, '3.5%_timing'] = output3
        #    top_performers_clean.loc[rowIndex, '5%_timing'] = output5
            
            all_ranking.at[rowIndex, '2%'] = output2 # copy over time format
            all_ranking.at[rowIndex, '3.5%'] = output3
            all_ranking.at[rowIndex, '5%'] = output5
    #        all_ranking.at[rowIndex, '10%'] = output10

            
         #   print('output', output2, output3, output5)


        
        except:
            
            pass
                        
             


/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_49156/3128109130.py:69: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '04:39.061800' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  all_ranking.at[rowIndex, '2%'] = output2 # copy over time format
/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_49156/3128109130.py:70: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '04:43.165650' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  all_ranking.at[rowIndex, '3.5%'] = output3
/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_49156/3128109130.py:71: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '04:47.269500' has dtype incompatible with floa

In [99]:
all_ranking=final_tiered_selection.reset_index(drop=True)

In [100]:
all_ranking

,index,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT_x,DISTANCE,EVENT_CLASS,...,Delta3.5,Delta5,Delta_Benchmark,PERF_SCALAR,clean_name,DOB_parsed,TIER,DOB_new,year_extract,age_extract
0,296,Chua Joshua,10.76,,,3,,100m,,,...,-0.07880,0.0760,-0.44,0.736434,joshua chua,2004-02-25,Tier 5,2004-02-25,2004.0,21.0
1,338,"Danish Iftikhar Rosl, .",0:10.52,NJAC Malaysia,12,1,Open,Dash,100.0,None,...,0.16120,0.3160,-0.20,3.062016,danish iftikhar rosl,2007-01-01,Tier 3,2007-01-01,2007.0,18.0
2,443,Gan Ian,0:10.72,Wings Athletic Club,12,2,Open,Dash,100.0,None,...,-0.03880,0.1160,-0.40,1.124031,gan ian,2002-12-26,Tier 5,2002-12-26,2002.0,23.0
3,508,Ho Ann Heng Xander,10.68,,,1,,100m,,,...,0.00120,0.1560,-0.36,1.511628,xander ho ann heng,2000-05-19,Tier 4,2000-05-19,2000.0,25.0
4,509,Ho Ann Heng Xander Ann Heng,0:10.82,Singapore,12,7,Open,Dash,100.0,None,...,-0.13880,0.0160,-0.50,0.155039,ho xander ann heng,2000-05-19,Tier 5,2000-05-19,2000.0,25.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
104,7218,Loh Ding Rong Anson Ding Rong,17.50,<NA>,18.5,nan,<NA>,Shot Put,<NA>,<NA>,...,1.73190,1.9770,1.16,12.099143,anson loh ding rong,None,Tier 1,NaT,NaN,NaN
105,7266,Ng Gabriel,16.49,RI,18.5,2.0,A,Shot Put,,5.00kg,...,0.72190,0.9670,0.15,5.917993,ng gabriel,None,Tier 1,NaT,NaN,18.5
106,7582,Lee Gabriel Jin Yi,16.09,<NA>,18.5,3.0,<NA>,Triple Jump,<NA>,<NA>,...,0.68860,0.9280,0.13,5.814536,gabriel lee,2003-02-23,Tier 1,NaT,NaN,NaN
107,7671,Rozario Tia Louise,13.00,<NA>,18.5,5.0,<NA>,Triple Jump,<NA>,<NA>,...,0.79275,0.9825,0.35,7.766798,tia louise rozario,2000-10-14,Tier 1,NaT,NaN,NaN


In [101]:
all_ranking[['DOB', 'DOB_new']]

,DOB,DOB_new
0,2004-02-25 00:00:00,2004-02-25
1,2007-01-01,2007-01-01
2,2002-12-26,2002-12-26
3,2000-05-19 00:00:00,2000-05-19
4,2000-05-19,2000-05-19
...,...,...
104,<NA>,NaT
105,<NA>,NaT
106,23 Feb 03,NaT
107,14 Oct 00,NaT


In [102]:
os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/Asian Indoor/')

all_ranking.to_csv('asian_indoor_all_ranking_jan3.csv', encoding='utf-8')

# DOB Lookup

In [103]:
# Get list of names with no DOB

mask = ((all_ranking['DOB']=='None')|(all_ranking['DOB']=='nan'))

#mask = ((final_df['DOB'].isnull()))
#mask = final_df['DOB_new']=='NaT'

missing_names = all_ranking.loc[mask]

name_list = missing_names['NAME'].str.casefold().to_list()


In [104]:
name_list

['bharadwaj akash vijay']

In [105]:
all_ranking["DOB_new"] = pd.to_datetime(all_ranking["DOB_new"], errors="coerce", dayfirst=True)


In [106]:
# Create a list of name components to seach for permutations

#final_df['name_to_list'] = final_df['clean_name'].str.split(' ')
all_ranking.loc[:, 'name_to_list'] = all_ranking['clean_name'].str.split(' ')


In [107]:
all_ranking

,index,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT_x,DISTANCE,EVENT_CLASS,...,Delta5,Delta_Benchmark,PERF_SCALAR,clean_name,DOB_parsed,TIER,DOB_new,year_extract,age_extract,name_to_list
0,296,Chua Joshua,10.76,,,3,,100m,,,...,0.0760,-0.44,0.736434,joshua chua,2004-02-25,Tier 5,2004-02-25,2004.0,21.0,"[joshua, chua]"
1,338,"Danish Iftikhar Rosl, .",0:10.52,NJAC Malaysia,12,1,Open,Dash,100.0,None,...,0.3160,-0.20,3.062016,danish iftikhar rosl,2007-01-01,Tier 3,2007-01-01,2007.0,18.0,"[danish, iftikhar, rosl, ]"
2,443,Gan Ian,0:10.72,Wings Athletic Club,12,2,Open,Dash,100.0,None,...,0.1160,-0.40,1.124031,gan ian,2002-12-26,Tier 5,2002-12-26,2002.0,23.0,"[gan, ian]"
3,508,Ho Ann Heng Xander,10.68,,,1,,100m,,,...,0.1560,-0.36,1.511628,xander ho ann heng,2000-05-19,Tier 4,2000-05-19,2000.0,25.0,"[xander, ho, ann, heng]"
4,509,Ho Ann Heng Xander Ann Heng,0:10.82,Singapore,12,7,Open,Dash,100.0,None,...,0.0160,-0.50,0.155039,ho xander ann heng,2000-05-19,Tier 5,2000-05-19,2000.0,25.0,"[ho, xander, ann, heng]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
104,7218,Loh Ding Rong Anson Ding Rong,17.50,<NA>,18.5,nan,<NA>,Shot Put,<NA>,<NA>,...,1.9770,1.16,12.099143,anson loh ding rong,None,Tier 1,NaT,NaN,NaN,"[anson, loh, ding, rong]"
105,7266,Ng Gabriel,16.49,RI,18.5,2.0,A,Shot Put,,5.00kg,...,0.9670,0.15,5.917993,ng gabriel,None,Tier 1,NaT,NaN,18.5,"[ng, gabriel]"
106,7582,Lee Gabriel Jin Yi,16.09,<NA>,18.5,3.0,<NA>,Triple Jump,<NA>,<NA>,...,0.9280,0.13,5.814536,gabriel lee,2003-02-23,Tier 1,NaT,NaN,NaN,"[gabriel, lee]"
107,7671,Rozario Tia Louise,13.00,<NA>,18.5,5.0,<NA>,Triple Jump,<NA>,<NA>,...,0.9825,0.35,7.766798,tia louise rozario,2000-10-14,Tier 1,NaT,NaN,NaN,"[tia, louise, rozario]"


In [108]:
'''
# Original code
# Lookup DOB from name

all_ranking['dob_list'] = None  


for index, row in all_ranking.iterrows():
         
   # if str(row['DOB_new']) == 'NaT':
   # if ((final_df['DOB']=='None') or (final_df['DOB']=='nan')):
            
    name_components = row['name_to_list']
        
    permutations_list = []
    dob_list=[]
    string=''
    
    
    for p in permutations(name_components):
        
        string = " ".join(p)
        
        permutations_list.append(string)

        dob = dictionary_dob_clean_name.get(string)
        
        if dob is not None:
            dob_list.append(dob)

    
  #  null.at[index, 'permutations'] = permutations_list
   # athletes.loc[index, 'dob_list'] = dob_list
    all_ranking.at[index, 'dob_list'] = [dob_list] if isinstance(dob_list, list) else dob_list
'''


'\n# Original code\n# Lookup DOB from name\n\nall_ranking[\'dob_list\'] = None  \n\n\nfor index, row in all_ranking.iterrows():\n         \n   # if str(row[\'DOB_new\']) == \'NaT\':\n   # if ((final_df[\'DOB\']==\'None\') or (final_df[\'DOB\']==\'nan\')):\n            \n    name_components = row[\'name_to_list\']\n        \n    permutations_list = []\n    dob_list=[]\n    string=\'\'\n    \n    \n    for p in permutations(name_components):\n        \n        string = " ".join(p)\n        \n        permutations_list.append(string)\n\n        dob = dictionary_dob_clean_name.get(string)\n        \n        if dob is not None:\n            dob_list.append(dob)\n\n    \n  #  null.at[index, \'permutations\'] = permutations_list\n   # athletes.loc[index, \'dob_list\'] = dob_list\n    all_ranking.at[index, \'dob_list\'] = [dob_list] if isinstance(dob_list, list) else dob_list\n'

In [109]:
import pandas as pd
from itertools import permutations

all_ranking['dob_list'] = None

for index, row in all_ranking.iterrows():
    # Check if 'dob_new' is missing/null
    dob_new = row['DOB_new']
    if pd.isnull(dob_new) or dob_new in ('None', 'nan', ''):
        name_components = row['name_to_list']
        dob_list = []
        # Efficient: if name_components is not empty
        if name_components:
            for p in permutations(name_components):
                string = " ".join(p)
                dob = dictionary_dob_clean_name.get(string)
                if dob is not None:
                    dob_list.append(dob)
        # Save dob_list (empty if not found)
        all_ranking.at[index, 'dob_list'] = dob_list if dob_list else None
    else:
        # If dob_new is present, do not overwrite, just skip lookup
        all_ranking.at[index, 'dob_list'] = None


In [110]:
all_ranking

,index,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT_x,DISTANCE,EVENT_CLASS,...,Delta_Benchmark,PERF_SCALAR,clean_name,DOB_parsed,TIER,DOB_new,year_extract,age_extract,name_to_list,dob_list
0,296,Chua Joshua,10.76,,,3,,100m,,,...,-0.44,0.736434,joshua chua,2004-02-25,Tier 5,2004-02-25,2004.0,21.0,"[joshua, chua]",None
1,338,"Danish Iftikhar Rosl, .",0:10.52,NJAC Malaysia,12,1,Open,Dash,100.0,None,...,-0.20,3.062016,danish iftikhar rosl,2007-01-01,Tier 3,2007-01-01,2007.0,18.0,"[danish, iftikhar, rosl, ]",None
2,443,Gan Ian,0:10.72,Wings Athletic Club,12,2,Open,Dash,100.0,None,...,-0.40,1.124031,gan ian,2002-12-26,Tier 5,2002-12-26,2002.0,23.0,"[gan, ian]",None
3,508,Ho Ann Heng Xander,10.68,,,1,,100m,,,...,-0.36,1.511628,xander ho ann heng,2000-05-19,Tier 4,2000-05-19,2000.0,25.0,"[xander, ho, ann, heng]",None
4,509,Ho Ann Heng Xander Ann Heng,0:10.82,Singapore,12,7,Open,Dash,100.0,None,...,-0.50,0.155039,ho xander ann heng,2000-05-19,Tier 5,2000-05-19,2000.0,25.0,"[ho, xander, ann, heng]",None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
104,7218,Loh Ding Rong Anson Ding Rong,17.50,<NA>,18.5,nan,<NA>,Shot Put,<NA>,<NA>,...,1.16,12.099143,anson loh ding rong,None,Tier 1,NaT,NaN,NaN,"[anson, loh, ding, rong]",[2008-01-03]
105,7266,Ng Gabriel,16.49,RI,18.5,2.0,A,Shot Put,,5.00kg,...,0.15,5.917993,ng gabriel,None,Tier 1,NaT,NaN,18.5,"[ng, gabriel]","[2008-07-24, 2008-01-03]"
106,7582,Lee Gabriel Jin Yi,16.09,<NA>,18.5,3.0,<NA>,Triple Jump,<NA>,<NA>,...,0.13,5.814536,gabriel lee,2003-02-23,Tier 1,NaT,NaN,NaN,"[gabriel, lee]","[2003-02-23, 2003-02-23]"
107,7671,Rozario Tia Louise,13.00,<NA>,18.5,5.0,<NA>,Triple Jump,<NA>,<NA>,...,0.35,7.766798,tia louise rozario,2000-10-14,Tier 1,NaT,NaN,NaN,"[tia, louise, rozario]","[2000-10-14, 2000-10-15]"


In [111]:
'''
# AI optimized code
# Lookup DOB from name
from tqdm import tqdm
tqdm.pandas()


def lookup_dob_optimized(name_components, dob_dict):
    """
    Returns first DOB found (stops searching after first match)
    Use this if you only need one DOB per athlete
    """
    for perm in permutations(name_components):
        name_string = " ".join(perm)
        dob = dob_dict.get(name_string)
        if dob is not None:
            return dob  # Return immediately on first match
    return None


# If you want progress tracking
all_ranking['dob_list'] = all_ranking['name_to_list'].progress_apply(
    lambda x: lookup_dob_optimized(x, dictionary_dob_clean_name)
)
'''

'\n# AI optimized code\n# Lookup DOB from name\nfrom tqdm import tqdm\ntqdm.pandas()\n\n\ndef lookup_dob_optimized(name_components, dob_dict):\n    """\n    Returns first DOB found (stops searching after first match)\n    Use this if you only need one DOB per athlete\n    """\n    for perm in permutations(name_components):\n        name_string = " ".join(perm)\n        dob = dob_dict.get(name_string)\n        if dob is not None:\n            return dob  # Return immediately on first match\n    return None\n\n\n# If you want progress tracking\nall_ranking[\'dob_list\'] = all_ranking[\'name_to_list\'].progress_apply(\n    lambda x: lookup_dob_optimized(x, dictionary_dob_clean_name)\n)\n'

In [112]:
'''
from itertools import permutations

def find_dob_fast(name_components, dob_dict, max_variants=10):
    # Build likely variants: original, reversed, joined without spaces
    variants = [" ".join(name_components), " ".join(reversed(name_components))]
    # Optionally, add joined name without spaces
    variants.append("".join(name_components))
    # If more variants needed, add up to max_variants permutations (generator)
    if len(name_components) > 2 and max_variants > 2:
        perm_gen = permutations(name_components)
        for idx, p in enumerate(perm_gen):
            if idx+3 >= max_variants:
                break
            string = " ".join(p)
            if string not in variants:
                variants.append(string)
    # Search for DOB in dictionary and return first match
    for variant in variants:
        dob = dob_dict.get(variant)
        if dob:
            return dob
    return None

all_ranking['dob_list'] = None

for idx, row in all_ranking.iterrows():
    name_components = row['name_to_list']
    dob = find_dob_fast(name_components, dictionary_dob_clean_name)
    all_ranking.at[idx, 'dob_list'] = dob
'''

'\nfrom itertools import permutations\n\ndef find_dob_fast(name_components, dob_dict, max_variants=10):\n    # Build likely variants: original, reversed, joined without spaces\n    variants = [" ".join(name_components), " ".join(reversed(name_components))]\n    # Optionally, add joined name without spaces\n    variants.append("".join(name_components))\n    # If more variants needed, add up to max_variants permutations (generator)\n    if len(name_components) > 2 and max_variants > 2:\n        perm_gen = permutations(name_components)\n        for idx, p in enumerate(perm_gen):\n            if idx+3 >= max_variants:\n                break\n            string = " ".join(p)\n            if string not in variants:\n                variants.append(string)\n    # Search for DOB in dictionary and return first match\n    for variant in variants:\n        dob = dob_dict.get(variant)\n        if dob:\n            return dob\n    return None\n\nall_ranking[\'dob_list\'] = None\n\nfor idx, row in a

In [113]:
# Create a new column with the first element of dob_list (or NaN if empty)

# all_ranking['dob_first'] = all_ranking['dob_list'].apply(lambda x: x[0] if isinstance(x, list) and len(x) > 0 else None)


In [114]:
'''
# Filter athletes born between 1 Jan 2008 to 31 Dec 2009

# First, convert to datetime
final_df['dob_first'] = pd.to_datetime(final_df['dob_first'], errors='coerce')

# Then remove timezone if it exists
if final_df['dob_first'].dt.tz is not None:
    
    final_df['dob_first'] = final_df['dob_first'].dt.tz_localize(None)

start = datetime.datetime(2008, 1, 1)

end = datetime.datetime(2009, 12, 31)

start_date = np.datetime64(start)
end_date = np.datetime64(end)


mask = ((final_df['dob_first'] >= start_date) & (final_df['dob_first'] <= end_date)|(final_df['dob_first'].isna()))
athletes_selected = final_df.loc[mask]

'''

"\n# Filter athletes born between 1 Jan 2008 to 31 Dec 2009\n\n# First, convert to datetime\nfinal_df['dob_first'] = pd.to_datetime(final_df['dob_first'], errors='coerce')\n\n# Then remove timezone if it exists\nif final_df['dob_first'].dt.tz is not None:\n    \n    final_df['dob_first'] = final_df['dob_first'].dt.tz_localize(None)\n\nstart = datetime.datetime(2008, 1, 1)\n\nend = datetime.datetime(2009, 12, 31)\n\nstart_date = np.datetime64(start)\nend_date = np.datetime64(end)\n\n\nmask = ((final_df['dob_first'] >= start_date) & (final_df['dob_first'] <= end_date)|(final_df['dob_first'].isna()))\nathletes_selected = final_df.loc[mask]\n\n"

In [115]:
# Create dob_final column as definitive dob column

def select_dob_final(row):
    dob_new = row['DOB_new']
    dob_list = row['dob_list']
    
    if pd.notnull(dob_new) and dob_new not in ('None', 'nan', ''):
        return dob_new
    else:
        # If dob_list is a list and not empty, take the first value
        if isinstance(dob_list, list) and dob_list:
            return dob_list[0]
        else:
            return None

all_ranking['dob_final'] = all_ranking.apply(select_dob_final, axis=1)




In [116]:
# First, convert to datetime
all_ranking['dob_final'] = pd.to_datetime(all_ranking['dob_final'], errors='coerce')

# Then remove timezone if it exists
if all_ranking['dob_final'].dt.tz is not None:
    
    all_ranking['dob_final'] = all_ranking['dob_first'].dt.tz_localize(None)


In [119]:
all_ranking.head(50)

,index,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT_x,DISTANCE,EVENT_CLASS,...,PERF_SCALAR,clean_name,DOB_parsed,TIER,DOB_new,year_extract,age_extract,name_to_list,dob_list,dob_final
0,296,Chua Joshua,10.76,,,3,,100m,,,...,0.736434,joshua chua,2004-02-25,Tier 5,2004-02-25,2004.0,21.0,"[joshua, chua]",None,2004-02-25
1,338,"Danish Iftikhar Rosl, .",0:10.52,NJAC Malaysia,12,1,Open,Dash,100.0,None,...,3.062016,danish iftikhar rosl,2007-01-01,Tier 3,2007-01-01,2007.0,18.0,"[danish, iftikhar, rosl, ]",None,2007-01-01
2,443,Gan Ian,0:10.72,Wings Athletic Club,12,2,Open,Dash,100.0,None,...,1.124031,gan ian,2002-12-26,Tier 5,2002-12-26,2002.0,23.0,"[gan, ian]",None,2002-12-26
3,508,Ho Ann Heng Xander,10.68,,,1,,100m,,,...,1.511628,xander ho ann heng,2000-05-19,Tier 4,2000-05-19,2000.0,25.0,"[xander, ho, ann, heng]",None,2000-05-19
4,509,Ho Ann Heng Xander Ann Heng,0:10.82,Singapore,12,7,Open,Dash,100.0,None,...,0.155039,ho xander ann heng,2000-05-19,Tier 5,2000-05-19,2000.0,25.0,"[ho, xander, ann, heng]",None,2000-05-19
5,572,Jaiganth Laavinia,11.98,,,1,U20,100m,,,...,2.079038,jaiganth laavinia,None,Tier 4,NaT,NaN,NaN,"[jaiganth, laavinia]",[2006-01-22],2006-01-22
6,797,"Lee Joshua, Shyen",10.54,,,5,U18,100m,,,...,2.868217,lee joshua shyen,None,Tier 4,NaT,NaN,NaN,"[lee, joshua, shyen]","[2008-12-09, 2008-12-09, 2008-12-09]",2008-12-09
7,798,Lee Mark Ren,0:10.73,Oldham Athletics,12,1,Open,Dash,100.0,None,...,1.027132,lee mark,2004-04-17,Tier 5,2004-04-17,2004.0,21.0,"[lee, mark]",None,2004-04-17
8,879,Lim Yee Chern Clara,00:12.12,RI,18.5,1.0,A,100m,,,...,0.876289,lim yee chern clara,None,Tier 5,NaT,NaN,18.5,"[lim, yee, chern, clara]",None,NaT
9,955,Louis Marc Brian,10.31,,,1,,100m,,,...,5.096899,marc brian louis,2002-07-08,Tier 1,2002-07-08,2002.0,23.0,"[marc, brian, louis]",None,2002-07-08


In [120]:
os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/Asian Indoor/')

#all_ranking.to_csv('all_ranking_pre_jan3.csv', encoding='utf-8')

all_ranking.to_excel('all_ranking_pre_jan3.xlsx')

In [384]:
# Determine exact age of athletes as at Dec 31 2025
# If athletes has no null value for age then he/she will be filtered out hence check for missing DOBs

end = pd.Timestamp('2026-12-31')

# 2) Force DOB column to naive datetime (cast to string first to strip any tz info)
dob = pd.to_datetime(all_ranking['dob_final'].astype(str), errors='coerce')

# 3) Compute age in years (days / 365.25)
all_ranking['age_dec26'] = (end - dob).dt.days / 365.25

# 4) Filter: >19 and <20 years old on 2025-12-31
#mask = (age_years > 19) & (age_years < 20)
#athletes = athletes.loc[mask].copy()
#athletes['age_years'] = age_years[mask]


In [385]:
all_ranking

,index,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT_x,DISTANCE,EVENT_CLASS,...,clean_name,DOB_parsed,TIER,DOB_new,year_extract,age_extract,name_to_list,dob_list,dob_final,age_dec26
0,276,Chua Joshua,10.76,None,None,3,None,100m,None,None,...,joshua chua,2004-02-25,Tier 5,2004-02-25,2004.0,21.0,"[joshua, chua]",None,2004-02-25,22.847365
1,416,"Gan, Ian",10.82,Wings Athletics Club,12,3,Open,Dash,100,None,...,gan ian,2002-12-26,Tier 5,2002-12-26,2002.0,23.0,"[gan, ian]",None,2002-12-26,24.013689
2,472,Ho Ann Heng Xander,10.68,None,None,1,None,100m,None,None,...,xander ho ann heng,2000-05-19,Tier 4,2000-05-19,2000.0,25.0,"[xander, ho, ann, heng]",None,2000-05-19,26.617385
3,743,Lee Mark Ren,10.73,Oldham Athletics,12,1,Open,Dash,100,None,...,lee mark,2004-04-17,Tier 5,2004-04-17,2004.0,21.0,"[lee, mark]",None,2004-04-17,22.704997
4,822,Lim Yee Chern Clara,00:12.12,RI,18.5,1.0,A,100m,None,None,...,lim yee chern clara,None,Tier 5,NaT,NaN,18.5,"[lim, yee, chern, clara]",None,NaT,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
93,6707,Loh Ding Rong Anson,18.59,None,None,3,None,Shot Put,None,5kg,...,anson loh,2008-11-25,Tier 1,2008-11-25,2008.0,17.0,"[anson, loh]",None,2008-11-25,18.097194
94,6743,Ng Gabriel,16.49,RI,18.5,2.0,A,Shot Put,None,5.00kg,...,ng gabriel,None,Tier 1,NaT,NaN,18.5,"[ng, gabriel]","[2008-07-24, 2008-11-25]",2008-07-24,18.436687
95,7025,Lee Gabriel Jin Yi,15.89,nan,nan,nan,nan,Triple Jump,nan,nan,...,gabriel lee,2003-02-23,Tier 2,NaT,NaN,NaN,"[gabriel, lee]","[2003-02-23, 2003-02-23]",2003-02-23,23.852156
96,7114,Rozario Tia Louise,12.90m,FAC,12,1,Open,Triple Jump,0,None,...,rozario tia louise,2000-10-14,Tier 1,NaT,NaN,NaN,"[rozario, tia, louise]","[2000-10-15, 2000-10-14]",2000-10-15,26.209446


In [386]:
# Choose athletes born between 1 Jan 2008 to 31 Dec 2009
# Null values retained for verification

final_selection = all_ranking[(all_ranking['age_dec26']>=16) & (all_ranking['age_dec26']<=17) | (all_ranking['age_dec26'].isna())]

final_selection

,index,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT_x,DISTANCE,EVENT_CLASS,...,clean_name,DOB_parsed,TIER,DOB_new,year_extract,age_extract,name_to_list,dob_list,dob_final,age_dec26
4,822,Lim Yee Chern Clara,00:12.12,RI,18.5,1.0,A,100m,None,None,...,lim yee chern clara,None,Tier 5,NaT,NaN,18.5,"[lim, yee, chern, clara]",None,NaT,NaN
34,3089,Aidan Michael Soh Zi Ren,00:50.10,ACS(I),18.5,1.0,A,400m,None,None,...,aidan michael soh zi ren,None,Tier 5,NaT,NaN,18.5,"[aidan, michael, soh, zi, ren]",None,NaT,NaN
37,3143,Bharadwaj Akash Vijay,49.65,Singapore Management Universit,12,1,Open,Dash,400,None,...,bharadwaj akash vijay,None,Tier 4,NaT,NaN,NaN,"[bharadwaj, akash, vijay]",None,NaT,NaN
75,4820,Kimberly Chew Pei Min,02:22.51,RI,18.5,1.0,A,800m,None,None,...,kimberly chew pei min,None,Tier 5,NaT,NaN,18.5,"[kimberly, chew, pei, min]",None,NaT,NaN
86,5552,Tan Shou Ri Yei (Chen Shouyi),2.04,RI,18.5,1.0,A,High Jump,None,None,...,tan shou yi rei chen shouyi,None,Tier 5,NaT,NaN,18.5,"[tan, shou, yi, rei, chen, shouyi]",None,NaT,NaN


In [414]:
os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/Asian Indoor/')

final_selection.to_csv('final_selection.csv', encoding='utf-8')